# KuaiRand 即時推薦系統（Colab 訓練版）

1. 選單 **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**

這個 notebook 會跑完所有離線實驗（retrieval、ranking、real-time features），並產生 serving 需要的 `artifacts/`。最後一格會下載 `kuairand-recsys-results.zip`：完整 repo + 填好結果的 README + artifacts。

Serving（Kafka、Redis、API、demo 網頁）需要 Docker，在 Colab 跑不了。下載後在自己電腦解壓，執行 `./demo.sh`。

> Colab 關閉後檔案會消失，記得下載。

In [ ]:
import os
os.makedirs('/content/kuairand-recsys/serving', exist_ok=True)
os.makedirs('/content/kuairand-recsys/serving/static', exist_ok=True)
os.makedirs('/content/kuairand-recsys/tests', exist_ok=True)
%cd /content/kuairand-recsys

## 1. 寫入程式碼

In [ ]:
%%writefile .gitignore
data/
checkpoints/
artifacts/
artifacts_synth/
results_synth/
__pycache__/
*.pyc
.venv/
*.tar.gz
.DS_Store


In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app
ENV PYTHONPATH=/app PYTHONUNBUFFERED=1

# CPU-only PyTorch keeps the image small; inference here is a few ms per request
RUN pip install --no-cache-dir torch --index-url https://download.pytorch.org/whl/cpu
COPY serving/requirements.txt serving/requirements.txt
RUN pip install --no-cache-dir -r serving/requirements.txt

COPY *.py ./
COPY serving ./serving
COPY tests ./tests

CMD ["uvicorn", "serving.app:app", "--host", "0.0.0.0", "--port", "8000"]


In [ ]:
%%writefile README.md
# Real-Time Two-Stage Recommender on KuaiRand

An end-to-end short-video recommender: **two-tower retrieval** and a **deep CTR ranker** trained on [KuaiRand-Pure](https://kuairand.com/) (Kuaishou), served by a **real-time stack** in which user actions stream through **Kafka** into a **Redis feature store** and re-rank the very next feed. Real-time features are defined once and shared by offline training and online serving, so there is no training/serving skew, and a parity test checks it.

![Live demo: clicking videos streams events through Kafka into Redis, and the next feed re-ranks](docs/demo.gif)

*Clicking a video sends an event through Kafka into the Redis feature store; the next request re-ranks the feed (NEW = newly retrieved, ▲ = moved up).*

```mermaid
flowchart LR
    subgraph Offline["Offline (train)"]
        L[(KuaiRand logs)] --> P[Temporal split]
        P --> TT[Two-tower<br/>in-batch softmax + logQ]
        P --> BF[Point-in-time<br/>feature backfill]
        TT --> BF
        BF --> RK[Ranker<br/>+ real-time features]
    end
    subgraph Online["Online (serve)"]
        C[Client / demo] -- click --> API
        API -- event --> K[[Kafka]]
        K --> W[Feature worker]
        W --> R[(Redis<br/>feature store)]
        API -- "1 user vector + recent clicks" --> R
        API -- "2 ANN top-200" --> F[FAISS]
        API -- "3 features" --> R
        API -- "4 score" --> M[Ranker]
    end
    TT -. embeddings .-> F
    RK -. weights .-> M
    BF -. same FeatureLogic .-> W
```

## Results

<!-- RESULTS START -->
### Retrieval (test period, full-catalog ranking)

| Model | Recall@50 | NDCG@50 | Recall@100 | Recall@50 (cold users, n=209) |
|---|---|---|---|---|
| Popularity baseline | 0.1145 | 0.0433 | 0.1812 | 0.2075 |
| Two-tower, ID embeddings only | 0.1144 | 0.0434 | 0.1812 | 0.2094 |
| Two-tower, no logQ correction | 0.0524 | 0.0168 | 0.0891 | 0.0374 |
| **Two-tower (side features + logQ)** | 0.1780 | 0.0597 | 0.2865 | 0.2440 |

Two-tower vs popularity, Recall@50: **+55.4%**

### Ranking (test period, click prediction)

| Model | AUC | GAUC | LogLoss | AUC on random-exposure log |
|---|---|---|---|---|
| Item CTR baseline | 0.6742 | 0.6106 | 0.6370 | 0.5404 |
| **DNN** (used downstream) | 0.7377 | 0.6436 | 0.6011 | 0.6455 |
| DCN-v2 (DNN + cross layers) | 0.7285 | 0.6257 | 0.6035 | 0.6436 |

### Real-time features (DNN ranker, same data and seed)

| Ranker | Val GAUC | AUC | GAUC | LogLoss | AUC on random-exposure log |
|---|---|---|---|---|---|
| No real-time features | 0.6421 | 0.7370 | 0.6415 | 0.6008 | 0.6452 |
| + all real-time features | 0.6316 | 0.7454 | 0.6260 | 0.5969 | 0.6978 |
| + candidate-level only **(served)** | 0.6482 | 0.7389 | 0.6436 | 0.5899 | 0.6728 |
| + user-level only | 0.6303 | 0.7423 | 0.6263 | 0.6005 | 0.6720 |

Candidate-level features vary across the items in one request; user-level features are constant within a request. Served variant chosen by validation GAUC.

### Serving latency (`./demo.sh loadtest`, single machine, Docker)

| Concurrency | Real-time | QPS | p50 ms | p95 ms | p99 ms | Server: retrieval / features / ranking ms |
|---|---|---|---|---|---|---|
| 1 | yes | 158.1 | 5.14 | 10.22 | 19.76 | 0.22 / 1.3 / 1.81 |
| 8 | yes | 276.7 | 30.68 | 62.71 | 115.21 | 0.65 / 4.0 / 17.35 |
| 32 | yes | 214.6 | 102.06 | 475.27 | 810.8 | 0.99 / 6.55 / 39.03 |
| 8 | no | 215.9 | 27.89 | 75.68 | 222.0 | 0.88 / 1.07 / 27.6 |
<!-- RESULTS END -->

**Key findings**

- **logQ correction is the single biggest lever in retrieval.** Without it, in-batch negatives over-penalize popular videos and the model falls to half of the popularity baseline (Recall@50 0.052 vs. 0.115); with it, two-tower beats popularity by 55%.
- **ID embeddings alone collapse to popularity.** Most users have too little history for a per-user embedding; side features are what let similar users share signal.
- **Explicit feature crosses didn't pay off.** DCN-v2 trailed a plain DNN on this mostly low-cardinality feature set, so the DNN is used downstream.
- **The random-exposure log exposes exposure bias.** Item CTR is strong on the production log (0.674 AUC) but near chance on randomly shown videos (0.540); the learned ranker holds up (0.646).
- **Real-time features: AUC up, GAUC down, until diagnosed.** All seven features raised AUC and random-log AUC (0.645 → 0.698) but lowered GAUC. User-level signals are constant within a request, so they separate heavy clickers from light ones without reordering any one user's feed. Keeping only candidate-level features, chosen by validation GAUC, preserves per-user ranking.

## Modeling

**Data.** KuaiRand-Pure: ~27K users, ~7.5K candidate videos, ~1.4M exposures with click labels, plus user and video side features. It also contains a **random-exposure log** of videos shown at random rather than by Kuaishou's recommender, which makes an unbiased evaluation possible.

**Temporal split.** The last 4 days are test, the 3 days before are validation. A random split would let the model train on the future.

**Retrieval: two-tower.** Each tower combines an ID embedding with side-feature embeddings, an MLP, and an L2-normalized 64-d output scored by dot product, so item vectors can be precomputed into an ANN index.
- *In-batch negatives with sampled softmax*: other positives in the batch serve as negatives.
- *logQ correction*: popular items appear as in-batch negatives more often, so an uncorrected model learns to under-score them. Subtracting log(sampling probability) from each logit removes that bias ([Yi et al., RecSys 2019](https://research.google/pubs/sampling-bias-corrected-neural-modeling-for-large-corpus-item-recommendations/)).
- *Duplicate masking*: a second copy of the positive in the batch is not treated as a negative.

**Ranking.** Every categorical field (user, item, hour, tab, side features) is embedded and fed to a deep MLP, optionally with a DCN-v2 cross network in parallel ([Wang et al., WWW 2021](https://arxiv.org/abs/2008.13535)). On this feature set the explicit cross layers did not beat a plain DNN, so the DNN is the ranker used downstream.

**Evaluation.**
- Retrieval: **full-catalog** Recall@K and NDCG@K (no sampled negatives, which inflate numbers), split by warm and cold users.
- Ranking: **AUC, GAUC and LogLoss**, on both the standard log and the **random-exposure log**. The standard log only contains items the production system already chose; the random log tests ranking on items it would never have shown.
- Baselines: popularity for retrieval, smoothed per-item CTR for ranking.

## Real-time features

Seven short-term signals, all count-windowed (last N events) so a replayed log produces the same values as the original timeline:

| Feature | Meaning |
|---|---|
| `sim_to_recent_clicks` | cosine between the candidate and the mean embedding of the user's last 20 clicks |
| `tag_match_recent`, `author_match_recent` | share of recent clicks with the candidate's tag / author |
| `user_recent_ctr` | user's smoothed CTR over their last 50 exposures |
| `item_recent_ctr`, `item_recent_volume` | item's smoothed CTR and exposure count over its last 200 exposures |
| `user_recent_clicks` | how much short-term history exists |

**One definition, two backends.** `realtime_features.FeatureLogic` computes features from a store; offline it runs on an in-memory store, online on Redis. `tests/test_parity.py` replays the same events through both and asserts identical features.

**Point-in-time backfill.** To train with these features, `build_rt_features.py` merges the standard and random logs into one stream and replays it in event-time order, computing each exposure's features from the state *before* it, then recording it. No feature ever sees its own label or the future.

## Serving

- **Request path** (`serving/engine.py`): user vector from the two-tower, blended with the user's recent-click vector from Redis → FAISS top-200 (already-clicked items removed) → real-time features → ranker → top-20. Each stage is timed and returned with the response.
- **Write-time aggregation**: every candidate needs its item's CTR over the last 200 exposures. Reading those windows on each request meant moving ~40K values per call (p99 over 1 s at 32 concurrent clients). A Redis Lua script now maintains each window's click count and size atomically on write, so a request reads two numbers per candidate with one `HMGET` (43x faster reads); the parity test covers the aggregates too.
- **Streaming**: the API produces click events to Kafka keyed by user id, so each user's events stay ordered on one partition. The worker applies each poll batch to Redis in a single pipelined round trip and tracks Kafka→Redis lag.
- **Bootstrap**: before going live, the train+val history is replayed through the same feature logic and bulk-loaded into Redis, so the online store starts in exactly the state the model expects.
- **Load test**: closed-loop clients at several concurrency levels report QPS, p50/p95/p99 and the server-side stage breakdown.

## Run it

**1. Train** (local, or open `kuairand_recsys_colab.ipynb` in Colab and Run all)

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
./smoke_test.sh                      # whole pipeline on fake data, ~2 min

mkdir -p data && cd data
curl -L -O https://zenodo.org/records/10439422/files/KuaiRand-Pure.tar.gz
tar -xzf KuaiRand-Pure.tar.gz && cd ..
./run_all.sh                         # experiments + artifacts/ for serving
```

**2. Serve** (needs Docker)

```bash
./demo.sh            # Redis, Kafka, bootstrap, API, worker -> http://localhost:8000
./demo.sh replay     # stream the test period as live traffic
./demo.sh test       # online/offline feature parity test
./demo.sh loadtest   # latency benchmark, then: python make_report.py --update_readme
./demo.sh down
```

| Path | Purpose |
|---|---|
| `prepare_data.py` | ID/feature encoding, temporal split |
| `baselines.py` | Popularity and item-CTR baselines, eval helpers |
| `models.py` | `TwoTower` (in-batch softmax + logQ), `DCNv2` (with optional dense inputs) |
| `train_retrieval.py`, `train_ranking.py` | Offline experiments and ablations |
| `realtime_features.py` | Shared feature logic, in-memory and Redis stores |
| `build_rt_features.py` | Point-in-time backfill, embedding export |
| `train_rt_ranker.py` | Ranker with vs. without real-time features |
| `export_artifacts.py` | Bundles models and data for serving |
| `serving/` | FastAPI app, engine, Kafka worker, bootstrap, replayer, load test, demo page |
| `tests/test_parity.py` | Online/offline feature parity |

## Known limitations

- The ranker is trained on logged exposures but serves retrieval output, a different candidate distribution; end-to-end evaluation of the full pipeline is not done yet.
- The weight that blends recent clicks into the retrieval query (`RT_ALPHA`) is hand-set, not tuned offline.
- The demo uses the current hour and the most common app tab as request context.
- At 7.5K items exact search is already fast; ANN only matters at much larger catalogs.

## Roadmap

- [ ] **Sequential retrieval**: SASRec, then HSTU-style generative retrieval over the watch sequence
- [ ] **Multi-task ranking**: click, long view and like jointly (MMoE)
- [ ] **ANN trade-offs**: HNSW vs. exact search, recall vs. latency (`FAISS_INDEX=hnsw`)
- [ ] **Scale**: KuaiRand-27K (32M videos) with Spark preprocessing and sharded embeddings


In [ ]:
%%writefile baselines.py
"""Non-neural pieces: eval-set construction, feature matrices, and the two baselines.

Kept free of PyTorch so they can be tested and run on their own:
    python baselines.py --data_dir data/processed
"""
import argparse

import numpy as np

from common import load_processed, ranking_metrics, save_results, topk_metrics


# ---------------- retrieval ----------------
def retrieval_truth(inter, split):
    """Users with >=1 click in `split`, and the items they clicked."""
    pos = inter[(inter.split == split) & (inter.is_click == 1)]
    g = pos.groupby("uid")["iid"].unique()
    return g.index.to_numpy(), [np.asarray(v) for v in g.to_numpy()]


def train_click_counts(inter, n_items):
    train_pos = inter[(inter.split == "train") & (inter.is_click == 1)]
    counts = np.bincount(train_pos.iid.to_numpy(), minlength=n_items + 1).astype(np.float64)
    counts[0] = 0
    return counts


def popularity_topk(counts, k):
    order = np.argsort(-counts, kind="stable")
    return order[order != 0][:k]


def segment_users(inter, uids):
    """warm = clicked something in train; cold = no train clicks (side features carry the load)."""
    warm_set = set(inter.loc[(inter.split == "train") & (inter.is_click == 1), "uid"].unique())
    return np.array([u in warm_set for u in uids])


def eval_retrieval(topk_items, uids, truths, inter, ks):
    res = {"all": topk_metrics(topk_items, truths, ks), "n_users": int(len(uids))}
    warm = segment_users(inter, uids)
    for name, mask in [("warm", warm), ("cold", ~warm)]:
        if mask.sum() > 0:
            idx = np.where(mask)[0]
            res[name] = topk_metrics(topk_items[idx], [truths[i] for i in idx], ks)
            res[f"n_{name}"] = int(mask.sum())
    return res


# ---------------- ranking ----------------
def ranking_fields(meta):
    names = ["uid", "iid", "hour", "tab"] + [f"u_{n}" for n in meta["user_feat_names"]] + [f"i_{n}" for n in meta["item_feat_names"]]
    vocabs = [meta["n_users"] + 1, meta["n_items"] + 1, meta["hour_vocab"], meta["tab_vocab"]] + meta["user_vocabs"] + meta["item_vocabs"]
    return names, vocabs


def ranking_matrix(df, user_feats, item_feats):
    uid, iid = df.uid.to_numpy(), df.iid.to_numpy()
    base = np.stack([uid, iid, df.hour.to_numpy(), np.asarray(df.tab, dtype=np.int64)], axis=1)
    return np.concatenate([base, user_feats[uid], item_feats[iid]], axis=1).astype(np.int64)


def item_ctr_scores(train_df, eval_df, n_items, prior_strength=20.0):
    """Smoothed per-item CTR from the training period: the 'just show what gets clicked' baseline."""
    clicks = np.bincount(train_df.iid, weights=train_df.is_click, minlength=n_items + 1)
    imps = np.bincount(train_df.iid, minlength=n_items + 1)
    g = train_df.is_click.mean()
    ctr = (clicks + prior_strength * g) / (imps + prior_strength)
    return ctr[eval_df.iid.to_numpy()]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="data/processed")
    ap.add_argument("--results_dir", default="results")
    ap.add_argument("--ks", type=int, nargs="+", default=[20, 50, 100])
    args = ap.parse_args()

    meta, inter, rand, _, _ = load_processed(args.data_dir)

    # Popularity retrieval
    counts = train_click_counts(inter, meta["n_items"])
    uids, truths = retrieval_truth(inter, "test")
    top = popularity_topk(counts, max(args.ks))
    topk_items = np.tile(top, (len(uids), 1))
    pop = eval_retrieval(topk_items, uids, truths, inter, args.ks)
    print("Popularity retrieval (test):", {k: round(v, 4) for k, v in pop["all"].items()})
    save_results(args.results_dir, "retrieval_popularity", {"model": "popularity", "test": pop})

    # Item-CTR ranking baseline
    train = inter[inter.split == "train"]
    out = {"model": "item_ctr"}
    for name, df in [("test_standard", inter[inter.split == "test"]),
                     ("test_random", rand[rand.split == "test"] if rand is not None else None)]:
        if df is not None and len(df) and df.is_click.nunique() == 2:
            out[name] = ranking_metrics(df.uid.to_numpy(), df.is_click.to_numpy(), item_ctr_scores(train, df, meta["n_items"]))
            print(f"Item-CTR ranking ({name}):", {k: round(v, 4) if isinstance(v, float) else v for k, v in out[name].items()})
    save_results(args.results_dir, "ranking_item_ctr", out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile build_rt_features.py
"""Point-in-time backfill of real-time features for every logged exposure.

Standard and random-exposure logs are merged into one stream (that's what a user actually
experienced) and replayed in event-time order. For each exposure we compute features from the
state *before* it, then record it. A feature therefore never sees its own label or anything
from the future: the offline equivalent of what the online feature store would have returned.

Also exports the two-tower user/item embeddings (needed by the similarity feature and by serving).

    python build_rt_features.py
"""
import argparse
import os
import time

import numpy as np
import pandas as pd

from common import load_processed
from realtime_features import N_FEATURES, FeatureLogic, InMemoryStore


def export_embeddings(meta, user_feats, item_feats, ckpt, out_path):
    import torch
    from models import TwoTower

    state = torch.load(ckpt, map_location="cpu")
    model = TwoTower(meta["n_users"], meta["n_items"], user_feats, item_feats, meta["user_vocabs"], meta["item_vocabs"],
                     out_dim=state["user_tower.net.6.weight"].shape[0] if "user_tower.net.6.weight" in state else 64)
    model.load_state_dict(state)
    model.eval()
    with torch.no_grad():
        user_emb = model.user_tower(torch.arange(0, meta["n_users"] + 1)).numpy()
        item_emb = model.item_tower(torch.arange(0, meta["n_items"] + 1)).numpy()
    user_emb[0] = 0
    item_emb[0] = 0
    np.savez(out_path, user_emb=user_emb.astype(np.float32), item_emb=item_emb.astype(np.float32))
    print(f"Embeddings saved to {out_path}: users {user_emb.shape}, items {item_emb.shape}")


def merged_stream(inter, rand):
    parts = [inter.assign(src=0, row=np.arange(len(inter)))]
    if rand is not None:
        parts.append(rand.assign(src=1, row=np.arange(len(rand))))
    ev = pd.concat(parts, ignore_index=True)
    return ev.sort_values("time_ms", kind="mergesort").reset_index(drop=True)


def item_codes(meta, item_feats, name):
    names = meta["item_feat_names"]
    return item_feats[:, names.index(name)] if name in names else np.zeros(len(item_feats), dtype=np.int64)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="data/processed")
    ap.add_argument("--two_tower_ckpt", default="checkpoints/two_tower.pt")
    args = ap.parse_args()

    meta, inter, rand, user_feats, item_feats = load_processed(args.data_dir)
    emb_path = os.path.join(args.data_dir, "emb.npz")
    if os.path.exists(args.two_tower_ckpt):
        export_embeddings(meta, user_feats, item_feats, args.two_tower_ckpt, emb_path)
    elif not os.path.exists(emb_path):
        raise SystemExit(f"Need {args.two_tower_ckpt} (run train_retrieval.py first) or a precomputed {emb_path}")
    item_emb = np.load(emb_path)["item_emb"]

    logic = FeatureLogic(item_emb, item_codes(meta, item_feats, "tag_first"),
                         item_codes(meta, item_feats, "author_id"), meta["overall_ctr"])
    store = InMemoryStore()
    ev = merged_stream(inter, rand)
    uids, iids, clicks = ev.uid.to_numpy(), ev.iid.to_numpy(), ev.is_click.to_numpy()
    src, row = ev.src.to_numpy(), ev.row.to_numpy()

    out = [np.zeros((len(inter), N_FEATURES), np.float32),
           np.zeros((len(rand) if rand is not None else 0, N_FEATURES), np.float32)]
    t0 = time.time()
    for n in range(len(ev)):
        u, i = int(uids[n]), int(iids[n])
        out[src[n]][row[n]] = logic.compute(store, u, [i])[0]   # features BEFORE seeing this event
        logic.update(store, u, i, clicks[n])                    # then record it
        if (n + 1) % 200_000 == 0:
            print(f"  {n + 1:,}/{len(ev):,} events  ({time.time() - t0:.0f}s)")

    np.save(os.path.join(args.data_dir, "rt_standard.npy"), out[0])
    if rand is not None:
        np.save(os.path.join(args.data_dir, "rt_random.npy"), out[1])
    # the merged stream is also what the serving demo bootstraps and replays from
    np.savez(os.path.join(args.data_dir, "events.npz"), uid=uids, iid=iids, is_click=clicks,
             time_ms=ev.time_ms.to_numpy(), split=ev.split.to_numpy().astype(str))
    print(f"Backfilled {len(ev):,} events in {time.time() - t0:.0f}s -> {args.data_dir}/rt_*.npy")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile common.py
"""Shared helpers: data loading, device selection, metrics, result saving."""
import json
import os
import random

import numpy as np
import pandas as pd


def get_device():
    import torch
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    import torch
    torch.manual_seed(seed)


def load_processed(data_dir):
    with open(os.path.join(data_dir, "meta.json")) as f:
        meta = json.load(f)
    inter = pd.read_pickle(os.path.join(data_dir, "interactions.pkl"))
    rand_path = os.path.join(data_dir, "random.pkl")
    rand = pd.read_pickle(rand_path) if os.path.exists(rand_path) else None
    feats = np.load(os.path.join(data_dir, "features.npz"))
    return meta, inter, rand, feats["user_feats"], feats["item_feats"]


# ---------------- retrieval metrics ----------------
def topk_metrics(topk_items, truths, ks):
    """topk_items: [n_users, max_k] item ids ranked; truths: list of np arrays of relevant item ids.
    Full-ranking Recall@K and NDCG@K (no sampled negatives, which inflate numbers)."""
    out = {}
    for k in ks:
        discounts = 1.0 / np.log2(np.arange(2, k + 2))
        rec, ndcg = [], []
        for row, t in zip(topk_items, truths):
            hits = np.isin(row[:k], t)
            rec.append(hits.sum() / len(t))
            idcg = discounts[: min(len(t), k)].sum()
            ndcg.append((hits * discounts).sum() / idcg)
        out[f"recall@{k}"] = float(np.mean(rec)) if rec else 0.0
        out[f"ndcg@{k}"] = float(np.mean(ndcg)) if ndcg else 0.0
    return out


# ---------------- ranking metrics ----------------
def gauc(uids, labels, scores):
    """Impression-weighted per-user AUC over users that have both clicks and non-clicks.
    Closer to what matters in a feed: ranking quality *within* each user's candidates."""
    df = pd.DataFrame({"u": uids, "y": labels, "s": scores})
    df["r"] = df.groupby("u")["s"].rank(method="average")
    g = df.groupby("u").agg(n=("y", "size"), pos=("y", "sum"), rpos=("r", lambda r: r[df.loc[r.index, "y"] == 1].sum()))
    g = g[(g.pos > 0) & (g.pos < g.n)]
    if g.empty:
        return float("nan")
    neg = g.n - g.pos
    auc_u = (g.rpos - g.pos * (g.pos + 1) / 2) / (g.pos * neg)
    return float((auc_u * g.n).sum() / g.n.sum())


def ranking_metrics(uids, labels, scores):
    from sklearn.metrics import log_loss, roc_auc_score
    p = np.clip(scores, 1e-6, 1 - 1e-6)
    return {
        "auc": float(roc_auc_score(labels, scores)),
        "gauc": gauc(uids, labels, scores),
        "logloss": float(log_loss(labels, p)),
        "n": int(len(labels)),
    }


def save_results(results_dir, name, payload):
    os.makedirs(results_dir, exist_ok=True)
    path = os.path.join(results_dir, f"{name}.json")
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"Saved {path}")


In [ ]:
%%writefile demo.sh
#!/usr/bin/env bash
# Start the serving stack:  ./demo.sh            (Redis, Kafka, bootstrap, API, worker)
#                           ./demo.sh replay     (also stream the test-period log as live traffic)
#                           ./demo.sh loadtest   (latency benchmark -> results/latency.json)
#                           ./demo.sh test       (online/offline feature parity test against Redis)
#                           ./demo.sh down
# Use ARTIFACTS=./artifacts_synth ./demo.sh to try the stack on the smoke-test artifacts.
set -euo pipefail
cd "$(dirname "$0")"
ART="${ARTIFACTS:-./artifacts}"

case "${1:-up}" in
  up)
    [ -f "$ART/ranker_rt.pt" ] || { echo "No artifacts in $ART. Run ./run_all.sh (or the Colab notebook) first."; exit 1; }
    mkdir -p results
    docker compose build
    docker compose up -d --wait redis kafka
    docker compose run --rm bootstrap
    docker compose up -d --wait api worker
    echo
    echo "Demo running: http://localhost:8000"
    echo "Live traffic:  ./demo.sh replay     Load test: ./demo.sh loadtest"
    ;;
  replay)
    docker compose --profile replay up -d replayer
    echo "Replaying test-period events. Turn on auto-refresh in the demo to watch feeds change."
    ;;
  loadtest)
    rm -f results/latency.json
    for c in 1 8 32; do
      docker compose run --rm --no-deps api python -m serving.loadtest --url http://api:8000 --concurrency "$c" --duration 20
    done
    docker compose run --rm --no-deps api python -m serving.loadtest --url http://api:8000 --concurrency 8 --duration 20 --no_realtime
    echo "Done. Run 'python make_report.py --update_readme' to add the latency table to README.md"
    ;;
  test)
    docker compose run --rm --no-deps api python tests/test_parity.py
    ;;
  down)
    docker compose --profile replay --profile tools down
    ;;
  *) echo "usage: $0 [up|replay|loadtest|test|down]"; exit 1 ;;
esac


In [ ]:
%%writefile docker-compose.yml
x-app: &app
  build: .
  image: kuairand-recsys:latest
  environment:
    REDIS_HOST: redis
    KAFKA_BOOTSTRAP: kafka:9092
    ARTIFACTS_DIR: /app/artifacts
    FAISS_INDEX: ${FAISS_INDEX:-flat}
    RT_ALPHA: ${RT_ALPHA:-1.0}
    REPLAY_RATE: ${REPLAY_RATE:-300}
  volumes:
    - ${ARTIFACTS:-./artifacts}:/app/artifacts:ro
    - ./results:/app/results

services:
  redis:
    image: redis:7-alpine
    ports: ["6379:6379"]
    healthcheck:
      test: ["CMD", "redis-cli", "ping"]
      interval: 3s
      retries: 20

  kafka:
    image: apache/kafka:3.8.0
    environment:
      KAFKA_NODE_ID: 1
      KAFKA_PROCESS_ROLES: broker,controller
      KAFKA_LISTENERS: PLAINTEXT://:9092,CONTROLLER://:9093
      KAFKA_ADVERTISED_LISTENERS: PLAINTEXT://kafka:9092
      KAFKA_CONTROLLER_LISTENER_NAMES: CONTROLLER
      KAFKA_LISTENER_SECURITY_PROTOCOL_MAP: CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      KAFKA_CONTROLLER_QUORUM_VOTERS: 1@kafka:9093
      KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR: 1
      KAFKA_TRANSACTION_STATE_LOG_MIN_ISR: 1
      KAFKA_GROUP_INITIAL_REBALANCE_DELAY_MS: 0
      KAFKA_NUM_PARTITIONS: 4
    healthcheck:
      test: ["CMD-SHELL", "/opt/kafka/bin/kafka-broker-api-versions.sh --bootstrap-server localhost:9092 > /dev/null 2>&1"]
      interval: 5s
      start_period: 10s
      retries: 30

  api:
    <<: *app
    command: ["sh", "-c", "uvicorn serving.app:app --host 0.0.0.0 --port 8000 --workers ${API_WORKERS:-2}"]
    ports: ["8000:8000"]
    restart: unless-stopped
    depends_on:
      redis: {condition: service_healthy}
      kafka: {condition: service_healthy}
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"]
      interval: 5s
      start_period: 30s
      retries: 20

  worker:
    <<: *app
    command: ["python", "-m", "serving.worker"]
    restart: unless-stopped
    depends_on:
      redis: {condition: service_healthy}
      kafka: {condition: service_healthy}

  replayer:
    <<: *app
    command: ["python", "-m", "serving.replayer"]
    profiles: ["replay"]
    depends_on:
      kafka: {condition: service_healthy}

  bootstrap:
    <<: *app
    command: ["python", "-m", "serving.bootstrap"]
    profiles: ["tools"]
    depends_on:
      redis: {condition: service_healthy}
      kafka: {condition: service_healthy}


In [ ]:
%%writefile export_artifacts.py
"""Bundle everything the serving stack needs into artifacts/ (mounted into the containers).

    python export_artifacts.py
"""
import argparse
import json
import os
import shutil

import numpy as np


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="data/processed")
    ap.add_argument("--ckpt_dir", default="checkpoints")
    ap.add_argument("--out_dir", default="artifacts")
    ap.add_argument("--n_demo_users", type=int, default=40)
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)

    for f in ["meta.json", "features.npz", "id_maps.npz", "items_meta.csv", "emb.npz", "events.npz"]:
        shutil.copy(os.path.join(args.data_dir, f), os.path.join(args.out_dir, f))
    for f in ["ranker_base.pt", "ranker_base.json", "ranker_rt.pt", "ranker_rt.json"]:
        shutil.copy(os.path.join(args.ckpt_dir, f), os.path.join(args.out_dir, f))

    # demo users: the most active users in the test period (they make the live demo interesting)
    ev = np.load(os.path.join(args.data_dir, "events.npz"))
    test = ev["split"] == "test"
    counts = np.bincount(ev["uid"][test & (ev["is_click"] == 1)])
    demo = [int(u) for u in np.argsort(-counts)[: args.n_demo_users] if u != 0 and counts[u] > 0]
    with open(os.path.join(args.out_dir, "demo_users.json"), "w") as f:
        json.dump(demo, f)

    # serving defaults for request context the demo doesn't have (which app tab the feed is on)
    import pandas as pd
    inter = pd.read_pickle(os.path.join(args.data_dir, "interactions.pkl"))
    with open(os.path.join(args.out_dir, "serving_config.json"), "w") as f:
        json.dump({"default_tab": int(inter.tab.mode().iloc[0])}, f)

    size = sum(os.path.getsize(os.path.join(args.out_dir, f)) for f in os.listdir(args.out_dir))
    print(f"Artifacts written to {args.out_dir}/ ({size / 1e6:.0f} MB, {len(demo)} demo users)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile make_report.py
"""Collect results/*.json into the Markdown tables for the README."""
import argparse
import glob
import json
import os

RETRIEVAL_ORDER = ["popularity", "two_tower_id_only", "two_tower_no_logq", "two_tower"]
RETRIEVAL_LABEL = {
    "popularity": "Popularity baseline",
    "two_tower_id_only": "Two-tower, ID embeddings only",
    "two_tower_no_logq": "Two-tower, no logQ correction",
    "two_tower": "**Two-tower (side features + logQ)**",
}
RANKING_ORDER = ["item_ctr", "dnn", "dcn"]
RANKING_LABEL = {"item_ctr": "Item CTR baseline", "dnn": "**DNN** (used downstream)", "dcn": "DCN-v2 (DNN + cross layers)"}


def fmt(x):
    return "–" if x is None else f"{x:.4f}"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results_dir", default="results")
    ap.add_argument("--update_readme", action="store_true", help="write the tables into README.md")
    args = ap.parse_args()
    res = {}
    for p in glob.glob(os.path.join(args.results_dir, "*.json")):
        with open(p) as f:
            res[os.path.basename(p)[:-5]] = json.load(f)

    n_cold = res.get("retrieval_two_tower", {}).get("test", {}).get("n_cold")
    cold_hdr = f"Recall@50 (cold users, n={n_cold})" if n_cold else "Recall@50 (cold users)"
    lines = ["### Retrieval (test period, full-catalog ranking)", "",
             f"| Model | Recall@50 | NDCG@50 | Recall@100 | {cold_hdr} |",
             "|---|---|---|---|---|"]
    for key in RETRIEVAL_ORDER:
        r = res.get(f"retrieval_{key}")
        if not r:
            continue
        t = r["test"]
        cold = t.get("cold", {}).get("recall@50")
        lines.append(f"| {RETRIEVAL_LABEL[key]} | {fmt(t['all'].get('recall@50'))} | {fmt(t['all'].get('ndcg@50'))} | "
                     f"{fmt(t['all'].get('recall@100'))} | {fmt(cold)} |")
    pop = res.get("retrieval_popularity", {}).get("test", {}).get("all", {}).get("recall@50")
    tt = res.get("retrieval_two_tower", {}).get("test", {}).get("all", {}).get("recall@50")
    if pop and tt:
        lines += ["", f"Two-tower vs popularity, Recall@50: **{(tt / pop - 1) * 100:+.1f}%**"]

    lines += ["", "### Ranking (test period, click prediction)", "",
              "| Model | AUC | GAUC | LogLoss | AUC on random-exposure log |", "|---|---|---|---|---|"]
    for key in RANKING_ORDER:
        r = res.get(f"ranking_{key}")
        if not r or "test_standard" not in r:
            continue
        s, rnd = r["test_standard"], r.get("test_random", {})
        lines.append(f"| {RANKING_LABEL[key]} | {fmt(s['auc'])} | {fmt(s['gauc'])} | {fmt(s['logloss'])} | {fmt(rnd.get('auc'))} |")

    rt = res.get("ranking_realtime")
    if rt and "variants" in rt:
        model = rt.get("model", "dnn").upper().replace("DCN", "DCN-v2")
        served = rt.get("served_variant")
        labels = {"base": "No real-time features", "all": "+ all real-time features",
                  "candidate": "+ candidate-level only", "user": "+ user-level only"}
        lines += ["", f"### Real-time features ({model} ranker, same data and seed)", "",
                  "| Ranker | Val GAUC | AUC | GAUC | LogLoss | AUC on random-exposure log |", "|---|---|---|---|---|---|"]
        for key, v in rt["variants"].items():
            s, rnd = v.get("test", {}), v.get("test_random", {})
            label = labels.get(key, key) + (" **(served)**" if key == served else "")
            lines.append(f"| {label} | {fmt(v['val'].get('gauc'))} | {fmt(s.get('auc'))} | {fmt(s.get('gauc'))} | "
                         f"{fmt(s.get('logloss'))} | {fmt(rnd.get('auc'))} |")
        lines += ["", "Candidate-level features vary across the items in one request; user-level features are constant "
                      "within a request. Served variant chosen by validation GAUC."]

    lat_path = os.path.join(args.results_dir, "latency.json")
    if os.path.exists(lat_path):
        with open(lat_path) as f:
            runs = json.load(f)
        lines += ["", "### Serving latency (`./demo.sh loadtest`, single machine, Docker)", "",
                  "| Concurrency | Real-time | QPS | p50 ms | p95 ms | p99 ms | Server: retrieval / features / ranking ms |",
                  "|---|---|---|---|---|---|---|"]
        for r in runs:
            s = r["server_stage_mean_ms"]
            lines.append(f"| {r['concurrency']} | {'yes' if r['realtime'] else 'no'} | {r['qps']} | {r['p50_ms']} | {r['p95_ms']} | "
                         f"{r['p99_ms']} | {s['retrieval']} / {s['features']} / {s['ranking']} |")

    report = "\n".join(lines)
    print(report)
    with open(os.path.join(args.results_dir, "RESULTS.md"), "w") as f:
        f.write(report + "\n")
    print(f"\nWritten to {args.results_dir}/RESULTS.md")

    if args.update_readme and os.path.exists("README.md"):
        with open("README.md") as f:
            readme = f.read()
        start, end = "<!-- RESULTS START -->", "<!-- RESULTS END -->"
        if start in readme and end in readme:
            readme = readme.split(start)[0] + start + "\n" + report + "\n" + end + readme.split(end)[1]
            with open("README.md", "w") as f:
                f.write(readme)
            print("README.md results section updated.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile models.py
"""Two-tower retrieval model and DCN-v2 ranking model."""
import torch
import torch.nn as nn
import torch.nn.functional as F


def mlp(in_dim, hidden, out_dim=None, dropout=0.1):
    layers, d = [], in_dim
    for h in hidden:
        layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
        d = h
    if out_dim is not None:
        layers.append(nn.Linear(d, out_dim))
        d = out_dim
    return nn.Sequential(*layers), d


# ============================ Retrieval ============================
class Tower(nn.Module):
    """ID embedding + categorical side-feature embeddings -> MLP -> L2-normalized vector.

    The side-feature table lives on the module as a buffer, so the tower takes only IDs.
    Side features are what let the model say something about users/items with little history.
    """

    def __init__(self, n_ids, feat_table, feat_vocabs, id_dim=64, feat_dim=16,
                 hidden=(256, 128), out_dim=64, dropout=0.1):
        super().__init__()
        self.id_emb = nn.Embedding(n_ids + 1, id_dim)
        self.register_buffer("feat_table", torch.as_tensor(feat_table, dtype=torch.long))
        self.feat_embs = nn.ModuleList([nn.Embedding(v, feat_dim) for v in feat_vocabs])
        in_dim = id_dim + feat_dim * len(feat_vocabs)
        self.net, _ = mlp(in_dim, hidden, out_dim, dropout)

    def forward(self, ids):
        parts = [self.id_emb(ids)]
        if len(self.feat_embs):
            f = self.feat_table[ids]
            parts += [emb(f[:, j]) for j, emb in enumerate(self.feat_embs)]
        return F.normalize(self.net(torch.cat(parts, dim=-1)), dim=-1)


class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, user_feats, item_feats, user_vocabs, item_vocabs,
                 use_side_features=True, **kw):
        super().__init__()
        if not use_side_features:
            user_feats, item_feats = user_feats[:, :0], item_feats[:, :0]
            user_vocabs, item_vocabs = [], []
        self.user_tower = Tower(n_users, user_feats, user_vocabs, **kw)
        self.item_tower = Tower(n_items, item_feats, item_vocabs, **kw)

    def in_batch_loss(self, uids, iids, temperature=0.05, log_q=None):
        """Sampled softmax with in-batch negatives.

        Every other positive item in the batch acts as a negative. Popular items show up as
        negatives more often, so the model learns to under-score them; logQ correction subtracts
        log(sampling prob) from each logit to undo that bias (Yi et al., RecSys 2019).
        """
        u = self.user_tower(uids)
        v = self.item_tower(iids)
        logits = u @ v.T / temperature
        if log_q is not None:
            logits = logits - log_q[iids].unsqueeze(0)
        # the same item can appear twice in a batch; don't treat a copy of the positive as a negative
        same_item = iids.unsqueeze(0) == iids.unsqueeze(1)
        eye = torch.eye(len(iids), dtype=torch.bool, device=iids.device)
        logits = logits.masked_fill(same_item & ~eye, float("-inf"))
        labels = torch.arange(len(iids), device=iids.device)
        return F.cross_entropy(logits, labels)


# ============================ Ranking ============================
class CrossLayer(nn.Module):
    """DCN-v2 cross layer: x_{l+1} = x0 * (W x_l + b) + x_l  (explicit bounded-degree feature crosses)."""

    def __init__(self, dim):
        super().__init__()
        self.lin = nn.Linear(dim, dim)

    def forward(self, x0, x):
        return x0 * self.lin(x) + x


class DCNv2(nn.Module):
    """Parallel DCN-v2 (Wang et al., WWW 2021). With use_cross=False it reduces to a plain DNN,
    which gives a clean ablation of what the explicit crosses add."""

    def __init__(self, field_vocabs, emb_dim=16, n_cross=3, hidden=(256, 128), dropout=0.1, use_cross=True, n_dense=0):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(v, emb_dim) for v in field_vocabs])
        self.n_dense = n_dense
        self.dense_norm = nn.BatchNorm1d(n_dense) if n_dense else None
        d0 = emb_dim * len(field_vocabs) + n_dense
        self.use_cross = use_cross
        self.cross = nn.ModuleList([CrossLayer(d0) for _ in range(n_cross)]) if use_cross else nn.ModuleList()
        self.deep, d_deep = mlp(d0, hidden, None, dropout)
        self.out = nn.Linear(d_deep + (d0 if use_cross else 0), 1)

    def forward(self, x, dense=None):
        parts = [emb(x[:, j]) for j, emb in enumerate(self.embs)]
        if self.n_dense:
            parts.append(self.dense_norm(dense))  # real-time features (continuous)
        x0 = torch.cat(parts, dim=-1)
        h = self.deep(x0)
        if self.use_cross:
            xc = x0
            for layer in self.cross:
                xc = layer(x0, xc)
            h = torch.cat([xc, h], dim=-1)
        return self.out(h).squeeze(-1)


In [ ]:
%%writefile prepare_data.py
"""Preprocess KuaiRand-Pure into model-ready arrays.

Outputs (in --out_dir):
  interactions.pkl  standard-feed exposures with uid/iid/hour/tab/is_click and a temporal split
  random.pkl        random-exposure log (unbiased; used for debiased evaluation of the ranker)
  features.npz      user_feats [n_users+1, F_u], item_feats [n_items+1, F_i]  (row 0 = unknown/padding)
  meta.json         vocab sizes, feature names, split dates, dataset stats

Split: temporal by date. The last --test_days dates are test, the --val_days before that are
validation, everything earlier is train. No random split -> no future leakage.
"""
import argparse
import glob
import json
import os

import numpy as np
import pandas as pd

USER_CAT_COLS = [
    "user_active_degree", "is_lowactive_period", "is_live_streamer", "is_video_author",
    "follow_user_num_range", "fans_user_num_range", "friend_user_num_range", "register_days_range",
] + [f"onehot_feat{i}" for i in range(18)]
ITEM_CAT_COLS = ["author_id", "video_type", "upload_type", "music_type", "tag_first", "duration_bucket"]
LOG_COLS = ["user_id", "video_id", "date", "hourmin", "time_ms", "is_click", "tab"]


def find(raw_dir, pattern):
    return sorted(glob.glob(os.path.join(raw_dir, "**", pattern), recursive=True))


def read_logs(files):
    if not files:
        return None
    frames = []
    for f in files:
        head = pd.read_csv(f, nrows=0).columns
        frames.append(pd.read_csv(f, usecols=[c for c in LOG_COLS if c in head]))
        print(f"  loaded {os.path.basename(f)}: {len(frames[-1]):,} rows")
    return pd.concat(frames, ignore_index=True)


def encode_column(series):
    """Map any column to contiguous ints starting at 1 (0 is reserved for unknown)."""
    codes, uniques = pd.factorize(series.astype(str))
    return codes + 1, len(uniques) + 1


def add_hour(df):
    if "hourmin" in df:
        df["hour"] = (df["hourmin"] // 100).clip(0, 23).astype(int) + 1
    else:
        df["hour"] = pd.to_datetime(df["time_ms"], unit="ms").dt.hour + 1
    return df


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw_dir", default="data/KuaiRand-Pure")
    ap.add_argument("--out_dir", default="data/processed")
    ap.add_argument("--val_days", type=int, default=3)
    ap.add_argument("--test_days", type=int, default=4)
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)

    std_files = find(args.raw_dir, "log_standard*.csv")
    rand_files = find(args.raw_dir, "log_random*.csv")
    uf_files = find(args.raw_dir, "user_features*.csv")
    vf_files = find(args.raw_dir, "video_features_basic*.csv")
    if not std_files or not uf_files or not vf_files:
        raise SystemExit(f"Could not find KuaiRand CSVs under {args.raw_dir}. See README for the download step.")

    print("Reading logs...")
    std = read_logs(std_files)
    rand = read_logs(rand_files)
    users = pd.read_csv(uf_files[0])
    videos = pd.read_csv(vf_files[0])

    # ---- ID maps (0 reserved) ----
    all_uids = pd.Index(sorted(set(users.user_id) | set(std.user_id) | (set(rand.user_id) if rand is not None else set())))
    all_iids = pd.Index(sorted(set(videos.video_id) | set(std.video_id) | (set(rand.video_id) if rand is not None else set())))
    n_users, n_items = len(all_uids), len(all_iids)

    # ---- user side features ----
    u_cols = [c for c in USER_CAT_COLS if c in users.columns]
    user_feats = np.zeros((n_users + 1, len(u_cols)), dtype=np.int64)
    u_rows = all_uids.get_indexer(users.user_id) + 1
    user_vocabs = []
    for j, c in enumerate(u_cols):
        codes, vocab = encode_column(users[c])
        user_feats[u_rows, j] = codes
        user_vocabs.append(vocab)

    # ---- item side features ----
    videos["tag_first"] = videos["tag"].astype(str).str.split(",").str[0] if "tag" in videos else "NA"
    if "video_duration" in videos:
        videos["duration_bucket"] = pd.qcut(videos["video_duration"], 10, labels=False, duplicates="drop").fillna(-1)
    i_cols = [c for c in ITEM_CAT_COLS if c in videos.columns]
    item_feats = np.zeros((n_items + 1, len(i_cols)), dtype=np.int64)
    i_rows = all_iids.get_indexer(videos.video_id) + 1
    item_vocabs = []
    for j, c in enumerate(i_cols):
        codes, vocab = encode_column(videos[c])
        item_feats[i_rows, j] = codes
        item_vocabs.append(vocab)

    # ---- temporal split ----
    dates = sorted(std.date.unique())
    test_dates = dates[-args.test_days:]
    val_dates = dates[-(args.test_days + args.val_days):-args.test_days]

    def to_split(d):
        return np.where(d.isin(test_dates), "test", np.where(d.isin(val_dates), "val", "train"))

    has_tab = "tab" in std.columns
    tab_vocab = 1
    if has_tab:
        tab_values = pd.concat([std.tab, rand.tab]) if (rand is not None and "tab" in rand) else std.tab
        tab_index = pd.Index(sorted(tab_values.unique()))
        tab_vocab = len(tab_index) + 1

    def finalize(df):
        df = add_hour(df.copy())
        out = pd.DataFrame({
            "uid": all_uids.get_indexer(df.user_id) + 1,
            "iid": all_iids.get_indexer(df.video_id) + 1,
            "hour": df.hour.astype(np.int64),
            "tab": (tab_index.get_indexer(df.tab) + 1) if (has_tab and "tab" in df) else 0,
            "date": df.date,
            "time_ms": df.time_ms if "time_ms" in df else 0,
            "is_click": df.is_click.astype(np.int64),
            "split": to_split(df.date),
        })
        return out.sort_values(["uid", "time_ms"]).reset_index(drop=True)

    inter = finalize(std)
    inter.to_pickle(os.path.join(args.out_dir, "interactions.pkl"))
    if rand is not None:
        finalize(rand).to_pickle(os.path.join(args.out_dir, "random.pkl"))
    np.savez(os.path.join(args.out_dir, "features.npz"), user_feats=user_feats, item_feats=item_feats)
    # raw-ID maps and human-readable item metadata, used by the serving demo
    np.savez(os.path.join(args.out_dir, "id_maps.npz"), raw_uids=all_uids.to_numpy(), raw_iids=all_iids.to_numpy())
    meta_cols = [c for c in ["video_id", "author_id", "tag_first", "video_type", "video_duration"] if c in videos.columns]
    items_meta = pd.DataFrame({"iid": np.arange(1, n_items + 1), "video_id": all_iids.to_numpy()})
    items_meta = items_meta.merge(videos[meta_cols], on="video_id", how="left")
    items_meta.to_csv(os.path.join(args.out_dir, "items_meta.csv"), index=False)

    stats = {s: {"rows": int((inter.split == s).sum()),
                 "clicks": int(((inter.split == s) & (inter.is_click == 1)).sum())}
             for s in ["train", "val", "test"]}
    meta = {
        "n_users": n_users, "n_items": n_items,
        "user_feat_names": u_cols, "user_vocabs": user_vocabs,
        "item_feat_names": i_cols, "item_vocabs": item_vocabs,
        "hour_vocab": 25, "tab_vocab": tab_vocab,
        "val_dates": [int(d) for d in val_dates], "test_dates": [int(d) for d in test_dates],
        "has_random_log": rand is not None,
        "stats": stats,
        "overall_ctr": float(inter.is_click.mean()),
    }
    with open(os.path.join(args.out_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print(f"\nUsers: {n_users:,}  Items: {n_items:,}  Standard-log exposures: {len(inter):,}  CTR: {meta['overall_ctr']:.3f}")
    print(f"Val dates: {meta['val_dates']}  Test dates: {meta['test_dates']}")
    for s, v in stats.items():
        print(f"  {s:5s}: {v['rows']:>9,} exposures, {v['clicks']:>8,} clicks")
    print(f"User features: {u_cols}\nItem features: {i_cols}")
    print(f"Saved to {args.out_dir}/")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile realtime_features.py
"""Real-time features, defined ONCE and used by both offline training and online serving.

Training/serving skew is one of the most common ways recommenders silently break: the
features a model sees in production are computed by different code than the features it
was trained on. Here both paths call the same `FeatureLogic`; only the storage differs:

    offline backfill  -> InMemoryStore (dicts of deques, replayed in event-time order)
    online serving    -> RedisStore    (Redis lists, updated by the Kafka consumer)

All windows are count-based (last N events), not wall-clock-based, so replaying a log at
any speed yields the same feature values as the original timeline.
"""
from collections import defaultdict, deque

import numpy as np

USER_CLICKS_N = 20       # last N clicked items per user
USER_EXPOSURES_N = 50    # last N exposures per user (click / no click)
ITEM_EXPOSURES_N = 200   # last N exposures per item

FEATURE_NAMES = [
    "user_recent_clicks",     # how much short-term history we have (0..1)
    "sim_to_recent_clicks",   # cosine(candidate, mean of recently clicked item embeddings)
    "tag_match_recent",       # share of recent clicks with the candidate's tag
    "author_match_recent",    # share of recent clicks from the candidate's author
    "user_recent_ctr",        # user's smoothed CTR over last exposures
    "item_recent_ctr",        # item's smoothed CTR over last exposures
    "item_recent_volume",     # how many recent exposures the item has (0..1)
]
N_FEATURES = len(FEATURE_NAMES)


# ============================ storage ============================
class InMemoryStore:
    """Newest-first lists, same ordering as Redis LPUSH."""

    def __init__(self):
        self.user_clicks = defaultdict(lambda: deque(maxlen=USER_CLICKS_N))
        self.user_exp = defaultdict(lambda: deque(maxlen=USER_EXPOSURES_N))
        self.item_exp = defaultdict(lambda: deque(maxlen=ITEM_EXPOSURES_N))

    def record(self, uid, iid, clicked):
        self.user_exp[uid].appendleft(int(clicked))
        self.item_exp[iid].appendleft(int(clicked))
        if clicked:
            self.user_clicks[uid].appendleft(int(iid))

    def record_many(self, events):
        for uid, iid, clicked in events:
            self.record(uid, iid, clicked)

    def get_user(self, uid):
        return list(self.user_clicks.get(uid, ())), list(self.user_exp.get(uid, ()))

    def get_items(self, iids):
        return [list(self.item_exp.get(int(i), ())) for i in iids]

    def get_item_stats(self, iids):
        """(clicks, exposures) over each item's window."""
        hist = [self.item_exp.get(int(i), ()) for i in iids]
        return np.array([sum(h) for h in hist], np.float64), np.array([len(h) for h in hist], np.float64)


# Item windows are read for every candidate on every request, so the online store keeps their
# aggregates up to date on write instead of returning full lists on read. The script pushes the
# new outcome, evicts the oldest one beyond the window, and adjusts the running click count
# atomically, so it stays exact with any number of workers.
ITEM_WINDOW_LUA = """
local n = redis.call('LPUSH', KEYS[1], ARGV[1])
local maxlen = tonumber(ARGV[2])
local delta = tonumber(ARGV[1])
if n > maxlen then
  delta = delta - tonumber(redis.call('RPOP', KEYS[1]))
  n = maxlen
end
if delta ~= 0 then redis.call('HINCRBY', KEYS[2], ARGV[3] .. ':s', delta) end
redis.call('HSET', KEYS[2], ARGV[3] .. ':n', n)
return n
"""


class RedisStore:
    """Same semantics as InMemoryStore, backed by Redis.

    user windows: lists (LPUSH + LTRIM), read in full (at most 70 values per request)
    item windows: list + running (clicks, count) in one hash, maintained by ITEM_WINDOW_LUA;
                  a request reads 2 numbers per candidate with a single HMGET
    """

    def __init__(self, client, prefix="rt"):
        self.r = client
        self.p = prefix

    def _k(self, kind, key):
        return f"{self.p}:{kind}:{key}"

    @property
    def _istat(self):
        return f"{self.p}:istat"

    def _record(self, pipe, uid, iid, clicked):
        self._push(pipe, self._k("uexp", uid), int(clicked), USER_EXPOSURES_N)
        pipe.eval(ITEM_WINDOW_LUA, 2, self._k("iexp", iid), self._istat, int(clicked), ITEM_EXPOSURES_N, int(iid))
        if clicked:
            self._push(pipe, self._k("uclk", uid), int(iid), USER_CLICKS_N)

    def record(self, uid, iid, clicked):
        pipe = self.r.pipeline(transaction=False)
        self._record(pipe, uid, iid, clicked)
        pipe.execute()

    @staticmethod
    def _push(pipe, key, value, maxlen):
        pipe.lpush(key, value)
        pipe.ltrim(key, 0, maxlen - 1)

    def record_many(self, events):
        """events: iterable of (uid, iid, clicked). One round trip for the whole batch."""
        pipe = self.r.pipeline(transaction=False)
        for uid, iid, clicked in events:
            self._record(pipe, uid, iid, clicked)
        pipe.execute()

    def get_user(self, uid):
        pipe = self.r.pipeline(transaction=False)
        pipe.lrange(self._k("uclk", uid), 0, -1)
        pipe.lrange(self._k("uexp", uid), 0, -1)
        clicks, exp = pipe.execute()
        return [int(x) for x in clicks], [int(x) for x in exp]

    def get_items(self, iids):
        pipe = self.r.pipeline(transaction=False)
        for i in iids:
            pipe.lrange(self._k("iexp", int(i)), 0, -1)
        return [[int(x) for x in v] for v in pipe.execute()]

    def get_item_stats(self, iids):
        fields = []
        for i in iids:
            fields += [f"{int(i)}:s", f"{int(i)}:n"]
        vals = self.r.hmget(self._istat, fields)
        v = np.array([float(x) if x is not None else 0.0 for x in vals])
        return v[0::2], v[1::2]

    def load_from(self, mem, batch=2000):
        """Bulk-load a backfilled InMemoryStore into Redis (feature-store bootstrap)."""
        pipe, n = self.r.pipeline(transaction=False), 0
        pipe.delete(self._istat)
        for kind, table in [("uclk", mem.user_clicks), ("uexp", mem.user_exp), ("iexp", mem.item_exp)]:
            for key, values in table.items():
                k = self._k(kind, key)
                pipe.delete(k)
                if values:
                    pipe.rpush(k, *values)  # values are newest-first; RPUSH keeps that order
                if kind == "iexp":
                    pipe.hset(self._istat, f"{key}:s", int(sum(values)))
                    pipe.hset(self._istat, f"{key}:n", len(values))
                n += 1
                if n % batch == 0:
                    pipe.execute()
        pipe.execute()
        return n


# ============================ features ============================
class FeatureLogic:
    def __init__(self, item_emb, item_tag, item_author, global_ctr):
        """item_emb: [n_items+1, d] L2-normalized (row 0 = padding); item_tag/item_author: [n_items+1] codes."""
        self.item_emb = np.asarray(item_emb, dtype=np.float32)
        self.item_tag = np.asarray(item_tag)
        self.item_author = np.asarray(item_author)
        self.g = float(global_ctr)

    def update(self, store, uid, iid, clicked):
        store.record(int(uid), int(iid), int(clicked))

    def recent_click_vector(self, clicks):
        if not clicks:
            return None
        m = self.item_emb[clicks].mean(0)
        n = np.linalg.norm(m)
        return m / n if n > 0 else None

    def compute(self, store, uid, iids, user_state=None):
        """Features for one user and a batch of candidate items -> float32 [len(iids), N_FEATURES].
        Pass `user_state` (from store.get_user) to avoid fetching it twice."""
        iids = np.asarray(iids, dtype=np.int64)
        clicks, exp = user_state if user_state is not None else store.get_user(int(uid))
        out = np.zeros((len(iids), N_FEATURES), dtype=np.float32)
        out[:, 0] = len(clicks) / USER_CLICKS_N
        m = self.recent_click_vector(clicks)
        if m is not None:
            out[:, 1] = self.item_emb[iids] @ m
            c = np.asarray(clicks)
            out[:, 2] = (self.item_tag[c][None, :] == self.item_tag[iids][:, None]).mean(1)
            out[:, 3] = (self.item_author[c][None, :] == self.item_author[iids][:, None]).mean(1)
        out[:, 4] = (sum(exp) + 5 * self.g) / (len(exp) + 5)
        clicks_i, n_i = store.get_item_stats(iids)
        out[:, 5] = (clicks_i + 20 * self.g) / (n_i + 20)
        out[:, 6] = n_i / ITEM_EXPOSURES_N
        return out


In [ ]:
%%writefile requirements.txt
torch>=2.1
numpy>=1.24
pandas>=2.0
scikit-learn>=1.3


In [ ]:
%%writefile run_all.sh
#!/usr/bin/env bash
# Full experiment on KuaiRand-Pure: preprocessing, baselines, two-tower (+ ablations), DCN-v2 (+ ablation), report.
set -euo pipefail
cd "$(dirname "$0")"

python prepare_data.py --raw_dir data/KuaiRand-Pure --out_dir data/processed
python baselines.py
python train_retrieval.py
python train_retrieval.py --no_side_features
python train_retrieval.py --no_logq
python train_ranking.py --model dcn
python train_ranking.py --model dnn
python build_rt_features.py
python train_rt_ranker.py
python export_artifacts.py
python make_report.py --update_readme


In [ ]:
%%writefile serving/__init__.py


In [ ]:
%%writefile serving/app.py
"""FastAPI service: recommendations, event ingestion, demo page.

GET  /recommend?uid=123&k=20&candidates=200&realtime=true
POST /event        {"uid": 123, "iid": 456, "clicked": true}   -> Kafka topic "events"
GET  /stats        stream-processing stats written by the worker
GET  /             interactive demo
"""
import json
import os
import time
from contextlib import asynccontextmanager

import redis
from fastapi import FastAPI, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel

from realtime_features import RedisStore
from serving.engine import Engine

ART_DIR = os.environ.get("ARTIFACTS_DIR", "artifacts")
REDIS_HOST = os.environ.get("REDIS_HOST", "localhost")
KAFKA = os.environ.get("KAFKA_BOOTSTRAP", "localhost:9092")
TOPIC = os.environ.get("EVENTS_TOPIC", "events")
STATIC = os.path.join(os.path.dirname(__file__), "static")

state = {}


@asynccontextmanager
async def lifespan(app):
    r = redis.Redis(host=REDIS_HOST, port=6379)
    state["redis"] = r
    state["engine"] = Engine(ART_DIR, RedisStore(r))
    try:
        from confluent_kafka import Producer
        state["producer"] = Producer({"bootstrap.servers": KAFKA, "linger.ms": 5})
    except Exception as e:  # recommendations still work without Kafka
        print(f"Kafka producer unavailable: {e}")
    yield
    if "producer" in state:
        state["producer"].flush(5)


app = FastAPI(title="KuaiRand two-stage recommender", lifespan=lifespan)


class Event(BaseModel):
    uid: int
    iid: int
    clicked: bool = True


@app.get("/health")
def health():
    e = state["engine"]
    return {"ok": True, "n_users": e.meta["n_users"], "n_items": e.meta["n_items"], "index": e.index_type}


@app.get("/recommend")
def recommend(uid: int, k: int = 20, candidates: int = 200, realtime: bool = True):
    try:
        return state["engine"].recommend(uid, k=k, n_candidates=candidates, realtime=realtime)
    except ValueError as e:
        raise HTTPException(400, str(e))


@app.post("/event")
def event(ev: Event):
    p = state.get("producer")
    if p is None:
        raise HTTPException(503, "Kafka producer not available")
    payload = {"uid": ev.uid, "iid": ev.iid, "clicked": int(ev.clicked), "sent_ms": time.time() * 1000, "source": "demo"}
    p.produce(TOPIC, key=str(ev.uid), value=json.dumps(payload))
    p.poll(0)
    return {"ok": True}


@app.get("/demo/users")
def demo_users():
    return {"users": state["engine"].demo_users}


@app.get("/stats")
def stats():
    r = state["redis"]
    vals = r.mget("rt:stats:events", "rt:stats:last_lag_ms", "rt:stats:lag_ema_ms", "rt:stats:bootstrap_events")
    f = lambda v: float(v) if v is not None else None
    return {"events_processed": f(vals[0]), "last_lag_ms": f(vals[1]), "lag_ema_ms": f(vals[2]), "bootstrapped_events": f(vals[3])}


@app.get("/")
def index():
    return FileResponse(os.path.join(STATIC, "index.html"))


In [ ]:
%%writefile serving/bootstrap.py
"""Feature-store bootstrap: rebuild online state from history, then hand off to streaming.

Replays the train+val period through the same record() logic as the offline backfill (in
memory, fast), bulk-loads the result into Redis, and creates the Kafka topic. After this the
store holds exactly the state the model would have seen at the start of the test period, and
the worker keeps it current from the stream.
"""
import os
import time

import numpy as np
import redis

from realtime_features import InMemoryStore, RedisStore

ART_DIR = os.environ.get("ARTIFACTS_DIR", "artifacts")
REDIS_HOST = os.environ.get("REDIS_HOST", "localhost")
KAFKA = os.environ.get("KAFKA_BOOTSTRAP", "localhost:9092")
TOPIC = os.environ.get("EVENTS_TOPIC", "events")


def create_topic():
    from confluent_kafka.admin import AdminClient, NewTopic
    admin = AdminClient({"bootstrap.servers": KAFKA})
    futures = admin.create_topics([NewTopic(TOPIC, num_partitions=4, replication_factor=1)])
    for t, f in futures.items():
        try:
            f.result()
            print(f"Created Kafka topic '{t}'")
        except Exception as e:
            print(f"Kafka topic '{t}': {e}")


def main():
    t0 = time.time()
    ev = np.load(os.path.join(ART_DIR, "events.npz"))
    hist = ev["split"] != "test"
    mem = InMemoryStore()
    mem.record_many(zip(ev["uid"][hist].tolist(), ev["iid"][hist].tolist(), ev["is_click"][hist].tolist()))
    print(f"Replayed {hist.sum():,} historical events in memory ({time.time() - t0:.1f}s)")

    r = redis.Redis(host=REDIS_HOST, port=6379)
    r.flushdb()
    n = RedisStore(r).load_from(mem)
    r.set("rt:stats:bootstrap_events", int(hist.sum()))
    r.set("rt:stats:events", 0)
    print(f"Loaded {n:,} keys into Redis ({time.time() - t0:.1f}s)")
    create_topic()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile serving/engine.py
"""The two-stage recommendation path, independent of the web framework.

    request -> user vector (+ real-time blend) -> ANN retrieval -> features -> ranker -> top-k

Each stage is timed so the API can report a latency breakdown.
"""
import csv
import json
import os
import time
from datetime import datetime

import numpy as np

from realtime_features import FeatureLogic

try:
    import faiss
except ImportError:  # local testing without faiss: exact search in numpy
    faiss = None


class Ranker:
    """Loads a ranker exported by train_rt_ranker.py and scores (X, dense) on CPU."""

    def __init__(self, art_dir, name):
        import torch

        from models import DCNv2

        torch.set_num_threads(int(os.environ.get("TORCH_THREADS", "1")))
        with open(os.path.join(art_dir, f"ranker_{name}.json")) as f:
            cfg = json.load(f)
        self.torch = torch
        self.model = DCNv2(cfg["vocabs"], emb_dim=cfg["emb_dim"], n_cross=cfg["n_cross"],
                           use_cross=cfg["use_cross"], n_dense=cfg["n_dense"])
        self.model.load_state_dict(torch.load(os.path.join(art_dir, f"ranker_{name}.pt"), map_location="cpu"))
        self.model.eval()
        self.uses_dense = cfg["n_dense"] > 0
        self.cols = cfg.get("rt_feature_idx")  # which real-time features this variant was trained on

    def score(self, X, dense):
        with self.torch.inference_mode():
            x = self.torch.from_numpy(X)
            if self.uses_dense and self.cols is not None:
                dense = np.ascontiguousarray(dense[:, self.cols])
            d = self.torch.from_numpy(dense) if self.uses_dense else None
            return self.torch.sigmoid(self.model(x, d)).numpy()


class Engine:
    def __init__(self, art_dir, store, rankers=None, index_type=None, rt_alpha=None):
        self.store = store
        with open(os.path.join(art_dir, "meta.json")) as f:
            self.meta = json.load(f)
        feats = np.load(os.path.join(art_dir, "features.npz"))
        self.user_feats, self.item_feats = feats["user_feats"], feats["item_feats"]
        emb = np.load(os.path.join(art_dir, "emb.npz"))
        self.user_emb, self.item_emb = emb["user_emb"], emb["item_emb"]
        self.n_items = self.meta["n_items"]
        with open(os.path.join(art_dir, "demo_users.json")) as f:
            self.demo_users = json.load(f)
        cfg_path = os.path.join(art_dir, "serving_config.json")
        self.default_tab = json.load(open(cfg_path))["default_tab"] if os.path.exists(cfg_path) else 0
        self.rt_alpha = float(rt_alpha if rt_alpha is not None else os.environ.get("RT_ALPHA", "1.0"))

        names = self.meta["item_feat_names"]
        col = lambda n: self.item_feats[:, names.index(n)] if n in names else np.zeros(len(self.item_feats), np.int64)
        self.logic = FeatureLogic(self.item_emb, col("tag_first"), col("author_id"), self.meta["overall_ctr"])

        self.items_meta = {}
        with open(os.path.join(art_dir, "items_meta.csv")) as f:
            for row in csv.DictReader(f):
                self.items_meta[int(row["iid"])] = row

        # ANN index over items 1..n (row r <-> item id r+1)
        self.index_type = index_type or os.environ.get("FAISS_INDEX", "flat")
        vecs = np.ascontiguousarray(self.item_emb[1:], dtype=np.float32)
        if faiss is None:
            self.index = None
        elif self.index_type == "hnsw":
            self.index = faiss.IndexHNSWFlat(vecs.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
            self.index.hnsw.efSearch = int(os.environ.get("HNSW_EF", "128"))
            self.index.add(vecs)
        else:
            self.index = faiss.IndexFlatIP(vecs.shape[1])
            self.index.add(vecs)
        self.rankers = rankers if rankers is not None else {"rt": Ranker(art_dir, "rt"), "base": Ranker(art_dir, "base")}

    # ---------------- stages ----------------
    def _retrieve(self, u, n):
        q = u.reshape(1, -1).astype(np.float32)
        if self.index is not None:
            scores, idx = self.index.search(q, n)
            return idx[0] + 1, scores[0]
        s = self.item_emb[1:] @ q[0]
        top = np.argpartition(-s, n)[:n] if n < len(s) else np.arange(len(s))
        top = top[np.argsort(-s[top])]
        return top + 1, s[top]

    def _ranking_matrix(self, uid, iids):
        n = len(iids)
        hour = datetime.now().hour + 1
        base = np.stack([np.full(n, uid), iids, np.full(n, hour), np.full(n, self.default_tab)], axis=1)
        return np.concatenate([base, np.repeat(self.user_feats[uid][None, :], n, 0), self.item_feats[iids]], axis=1).astype(np.int64)

    def item_info(self, iid):
        m = self.items_meta.get(int(iid), {})
        return {"iid": int(iid), "video_id": m.get("video_id"), "author_id": m.get("author_id"), "tag": m.get("tag_first")}

    def recommend(self, uid, k=20, n_candidates=200, realtime=True):
        t0 = time.perf_counter()
        uid = int(uid)
        if not 1 <= uid <= self.meta["n_users"]:
            raise ValueError(f"uid must be in 1..{self.meta['n_users']}")

        user_state = self.store.get_user(uid) if realtime else ([], [])
        clicks = user_state[0]
        u = self.user_emb[uid].copy()
        if realtime and clicks:
            # short-term interest: pull the user vector toward what they just clicked
            u = u + self.rt_alpha * self.logic.recent_click_vector(clicks)
            u /= np.linalg.norm(u) + 1e-12
        t1 = time.perf_counter()

        seen = set(clicks)
        iids, ret_scores = self._retrieve(u, min(n_candidates + len(seen), self.n_items))
        keep = np.array([i not in seen for i in iids], dtype=bool)
        iids, ret_scores = iids[keep][:n_candidates], ret_scores[keep][:n_candidates]
        t2 = time.perf_counter()

        X = self._ranking_matrix(uid, iids)
        dense = self.logic.compute(self.store, uid, iids, user_state) if realtime else np.zeros((len(iids), 0), np.float32)
        t3 = time.perf_counter()

        scores = self.rankers["rt" if realtime else "base"].score(X, dense)
        order = np.argsort(-scores)[:k]
        t4 = time.perf_counter()

        items = []
        for r in order:
            info = self.item_info(iids[r])
            info.update(score=round(float(scores[r]), 4), retrieval_rank=int(r) + 1)
            if realtime:
                info["features"] = {"sim_recent": round(float(dense[r, 1]), 3), "tag_match": round(float(dense[r, 2]), 3)}
            items.append(info)
        ms = lambda a, b: round((b - a) * 1000, 2)
        return {
            "uid": uid, "realtime": realtime, "items": items,
            "recent_clicks": [self.item_info(i) for i in clicks[:10]],
            "timings_ms": {"user_vector": ms(t0, t1), "retrieval": ms(t1, t2), "features": ms(t2, t3),
                           "ranking": ms(t3, t4), "total": ms(t0, t4)},
            "n_candidates": int(len(iids)),
        }


In [ ]:
%%writefile serving/loadtest.py
"""Closed-loop load test: N concurrent clients hammer /recommend for a fixed duration.

Reports throughput, client-side p50/p95/p99, and the server's per-stage breakdown.

    python -m serving.loadtest --url http://api:8000 --concurrency 16 --duration 30
"""
import argparse
import asyncio
import json
import os
import random
import time

import httpx
import numpy as np


async def client(http, url, uids, args, deadline, lat, stages, errors):
    while time.perf_counter() < deadline:
        uid = random.choice(uids)
        t = time.perf_counter()
        try:
            r = await http.get(f"{url}/recommend", params={"uid": uid, "k": 20, "candidates": args.candidates,
                                                          "realtime": str(not args.no_realtime).lower()})
            r.raise_for_status()
            lat.append((time.perf_counter() - t) * 1000)
            stages.append(r.json()["timings_ms"])
        except Exception:
            errors.append(1)


async def run(args):
    async with httpx.AsyncClient(timeout=10, limits=httpx.Limits(max_connections=args.concurrency)) as http:
        n_users = (await http.get(f"{args.url}/health")).json()["n_users"]
        uids = list(range(1, n_users + 1))
        for _ in range(20):  # warm-up
            await http.get(f"{args.url}/recommend", params={"uid": random.choice(uids)})
        lat, stages, errors = [], [], []
        start = time.perf_counter()
        deadline = start + args.duration
        await asyncio.gather(*[client(http, args.url, uids, args, deadline, lat, stages, errors) for _ in range(args.concurrency)])
        elapsed = time.perf_counter() - start

    lat = np.array(lat)
    res = {
        "concurrency": args.concurrency, "candidates": args.candidates, "realtime": not args.no_realtime,
        "requests": int(len(lat)), "errors": len(errors), "qps": round(len(lat) / elapsed, 1),
        "p50_ms": round(float(np.percentile(lat, 50)), 2), "p95_ms": round(float(np.percentile(lat, 95)), 2),
        "p99_ms": round(float(np.percentile(lat, 99)), 2),
        "server_stage_mean_ms": {k: round(float(np.mean([s[k] for s in stages])), 2) for k in stages[0]},
    }
    print(json.dumps(res, indent=2))
    if args.out:
        os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
        existing = json.load(open(args.out)) if os.path.exists(args.out) else []
        json.dump(existing + [res], open(args.out, "w"), indent=2)
        print(f"Appended to {args.out}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--url", default=os.environ.get("API_URL", "http://localhost:8000"))
    ap.add_argument("--concurrency", type=int, default=16)
    ap.add_argument("--duration", type=float, default=30)
    ap.add_argument("--candidates", type=int, default=200)
    ap.add_argument("--no_realtime", action="store_true")
    ap.add_argument("--out", default="results/latency.json")
    asyncio.run(run(ap.parse_args()))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile serving/replayer.py
"""Replays the test-period log into Kafka as live traffic.

    python -m serving.replayer --rate 300      # events per second
"""
import argparse
import json
import os
import time

import numpy as np
from confluent_kafka import Producer

ART_DIR = os.environ.get("ARTIFACTS_DIR", "artifacts")
KAFKA = os.environ.get("KAFKA_BOOTSTRAP", "localhost:9092")
TOPIC = os.environ.get("EVENTS_TOPIC", "events")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--rate", type=float, default=float(os.environ.get("REPLAY_RATE", "300")))
    ap.add_argument("--loop", action="store_true")
    args = ap.parse_args()

    ev = np.load(os.path.join(ART_DIR, "events.npz"))
    test = ev["split"] == "test"
    uids, iids, clicks = ev["uid"][test], ev["iid"][test], ev["is_click"][test]
    p = Producer({"bootstrap.servers": KAFKA, "linger.ms": 10})
    print(f"Replaying {len(uids):,} test-period events at {args.rate:.0f}/s", flush=True)

    batch = max(1, int(args.rate / 20))  # send in 50ms slices
    while True:
        start = time.time()
        for s in range(0, len(uids), batch):
            now = time.time() * 1000
            for u, i, c in zip(uids[s:s + batch], iids[s:s + batch], clicks[s:s + batch]):
                p.produce(TOPIC, key=str(int(u)), value=json.dumps(
                    {"uid": int(u), "iid": int(i), "clicked": int(c), "sent_ms": now, "source": "replay"}))
            p.poll(0)
            sent = s + batch
            ahead = sent / args.rate - (time.time() - start)
            if ahead > 0:
                time.sleep(ahead)
            if sent % (batch * 200) == 0:
                print(f"  sent {sent:,}/{len(uids):,}", flush=True)
        p.flush(10)
        if not args.loop:
            break
    print("Replay finished.", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile serving/requirements.txt
numpy>=1.26
fastapi>=0.110
uvicorn>=0.29
redis>=5.0
confluent-kafka>=2.3
faiss-cpu>=1.8
httpx>=0.27


In [ ]:
%%writefile serving/static/index.html
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Real-time Feed Demo</title>
<style>
  :root {
    --bg: #0e1116; --panel: #161b22; --line: #262d36; --text: #e6edf3; --muted: #8b949e;
    --accent: #f78166; --good: #3fb950; --info: #58a6ff;
  }
  * { box-sizing: border-box; }
  body { margin: 0; background: var(--bg); color: var(--text); font: 14px/1.45 ui-sans-serif, system-ui, -apple-system, "Segoe UI", sans-serif; }
  header { padding: 18px 24px; border-bottom: 1px solid var(--line); display: flex; align-items: baseline; gap: 16px; flex-wrap: wrap; }
  header h1 { font-size: 18px; margin: 0; letter-spacing: .2px; }
  header .sub { color: var(--muted); font-size: 13px; }
  main { display: grid; grid-template-columns: 300px 1fr; gap: 20px; padding: 20px 24px; }
  @media (max-width: 860px) { main { grid-template-columns: 1fr; padding: 16px; } }
  .panel { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; padding: 14px; margin-bottom: 16px; }
  .panel h2 { font-size: 12px; text-transform: uppercase; letter-spacing: .08em; color: var(--muted); margin: 0 0 10px; font-weight: 600; }
  .users { display: flex; flex-wrap: wrap; gap: 6px; }
  .chip { border: 1px solid var(--line); background: transparent; color: var(--text); border-radius: 999px; padding: 4px 10px; cursor: pointer; font: inherit; font-size: 12px; }
  .chip.active { background: var(--info); border-color: var(--info); color: #0e1116; font-weight: 600; }
  .row { display: flex; justify-content: space-between; align-items: center; padding: 4px 0; }
  .toggle { display: flex; align-items: center; gap: 8px; cursor: pointer; user-select: none; }
  .toggle input { accent-color: var(--good); width: 16px; height: 16px; }
  .muted { color: var(--muted); }
  .stat { font-variant-numeric: tabular-nums; font-weight: 600; }
  .bar { display: flex; height: 10px; border-radius: 5px; overflow: hidden; background: var(--line); margin: 8px 0 6px; }
  .bar span { display: block; height: 100%; }
  .legend { display: grid; grid-template-columns: 1fr auto; gap: 2px 10px; font-size: 12px; }
  .dot { display: inline-block; width: 8px; height: 8px; border-radius: 2px; margin-right: 6px; }
  .recent { display: flex; flex-wrap: wrap; gap: 6px; min-height: 24px; }
  .recent .tag { font-size: 12px; padding: 2px 8px; border-radius: 6px; }
  .feed { display: grid; grid-template-columns: repeat(auto-fill, minmax(170px, 1fr)); gap: 12px; }
  .card { background: var(--panel); border: 1px solid var(--line); border-radius: 10px; overflow: hidden; position: relative; transition: border-color .3s, transform .3s; }
  .card.new { border-color: var(--good); }
  .thumb { height: 92px; display: flex; align-items: flex-end; padding: 10px; font-weight: 700; font-size: 15px; color: #fff; text-shadow: 0 1px 2px rgba(0,0,0,.4); }
  .meta { padding: 10px; font-size: 12px; }
  .meta .line { display: flex; justify-content: space-between; color: var(--muted); }
  .rank { position: absolute; top: 8px; left: 8px; background: rgba(0,0,0,.55); color: #fff; border-radius: 6px; padding: 1px 7px; font-size: 12px; font-weight: 600; }
  .badge { position: absolute; top: 8px; right: 8px; border-radius: 6px; padding: 1px 7px; font-size: 11px; font-weight: 700; }
  .badge.new { background: var(--good); color: #0e1116; }
  .badge.up { background: var(--info); color: #0e1116; }
  .card button { width: 100%; border: 0; border-top: 1px solid var(--line); background: transparent; color: var(--accent); padding: 8px; cursor: pointer; font: inherit; font-weight: 600; }
  .card button:hover { background: rgba(247,129,102,.08); }
  .toast { position: fixed; bottom: 20px; right: 20px; background: var(--panel); border: 1px solid var(--line); border-radius: 8px; padding: 10px 14px; opacity: 0; transition: opacity .25s; }
  .toast.show { opacity: 1; }
  .err { color: var(--accent); }
</style>
</head>
<body>
<header>
  <h1>Two-stage recommender · live demo</h1>
  <span class="sub">Two-tower retrieval → real-time features (Kafka → Redis) → ranker</span>
</header>
<main>
  <aside>
    <div class="panel">
      <h2>User</h2>
      <div class="users" id="users"></div>
    </div>
    <div class="panel">
      <h2>Mode</h2>
      <label class="toggle row"><span>Real-time personalization</span><input type="checkbox" id="rt" checked></label>
      <label class="toggle row"><span>Auto-refresh (3s)</span><input type="checkbox" id="auto"></label>
    </div>
    <div class="panel">
      <h2>Recent clicks (from Redis)</h2>
      <div class="recent" id="recent"><span class="muted">—</span></div>
    </div>
    <div class="panel">
      <h2>Latency breakdown</h2>
      <div class="row"><span class="muted">Server total</span><span class="stat" id="total">—</span></div>
      <div class="bar" id="bar"></div>
      <div class="legend" id="legend"></div>
    </div>
    <div class="panel">
      <h2>Event stream</h2>
      <div class="row"><span class="muted">Events processed</span><span class="stat" id="ev">—</span></div>
      <div class="row"><span class="muted">Kafka → Redis lag</span><span class="stat" id="lag">—</span></div>
    </div>
  </aside>
  <section>
    <div class="feed" id="feed"></div>
  </section>
</main>
<div class="toast" id="toast"></div>
<script>
const COLORS = { user_vector: "#a371f7", retrieval: "#58a6ff", features: "#3fb950", ranking: "#f78166" };
let uid = null, prev = new Map();
const $ = (id) => document.getElementById(id);
const hue = (tag) => (parseInt(tag || "0", 10) * 47) % 360;
const grad = (tag) => `linear-gradient(135deg, hsl(${hue(tag)} 65% 42%), hsl(${(hue(tag) + 40) % 360} 70% 28%))`;

function toast(msg) { const t = $("toast"); t.textContent = msg; t.classList.add("show"); setTimeout(() => t.classList.remove("show"), 1600); }

async function loadUsers() {
  const { users } = await (await fetch("/demo/users")).json();
  $("users").innerHTML = users.map((u) => `<button class="chip" data-uid="${u}">${u}</button>`).join("");
  $("users").onclick = (e) => { if (e.target.dataset.uid) selectUser(+e.target.dataset.uid); };
  if (users.length) selectUser(users[0]);
}

function selectUser(u) {
  uid = u; prev = new Map();
  document.querySelectorAll(".chip").forEach((c) => c.classList.toggle("active", +c.dataset.uid === u));
  refresh();
}

async function refresh() {
  if (!uid) return;
  try {
    const r = await fetch(`/recommend?uid=${uid}&k=20&realtime=${$("rt").checked}`);
    if (!r.ok) throw new Error(await r.text());
    render(await r.json());
  } catch (e) { $("feed").innerHTML = `<p class="err">${e.message}</p>`; }
}

function render(d) {
  const first = prev.size === 0;
  $("feed").innerHTML = d.items.map((it, i) => {
    const before = prev.get(it.iid);
    let badge = "";
    if (!first && before === undefined) badge = `<span class="badge new">NEW</span>`;
    else if (!first && before > i) badge = `<span class="badge up">▲${before - i}</span>`;
    const f = it.features ? `<div class="line"><span>sim ${it.features.sim_recent}</span><span>tag-match ${it.features.tag_match}</span></div>` : "";
    return `<div class="card ${badge.includes("new") ? "new" : ""}">
      <div class="thumb" style="background:${grad(it.tag)}">#tag ${it.tag ?? "?"}</div>
      <span class="rank">${i + 1}</span>${badge}
      <div class="meta">
        <div class="line"><span>video ${it.video_id}</span><span>p=${it.score}</span></div>
        <div class="line"><span>author ${it.author_id}</span><span>retr #${it.retrieval_rank}</span></div>${f}
      </div>
      <button data-iid="${it.iid}">Watch ▶</button>
    </div>`;
  }).join("");
  prev = new Map(d.items.map((it, i) => [it.iid, i]));

  $("recent").innerHTML = d.recent_clicks.length
    ? d.recent_clicks.map((c) => `<span class="tag" style="background:${grad(c.tag)}">#${c.tag} · ${c.video_id}</span>`).join("")
    : `<span class="muted">${$("rt").checked ? "no recent clicks" : "real-time off"}</span>`;

  const t = d.timings_ms, parts = ["user_vector", "retrieval", "features", "ranking"];
  $("total").textContent = `${t.total.toFixed(1)} ms`;
  $("bar").innerHTML = parts.map((p) => `<span style="width:${(100 * t[p]) / Math.max(t.total, 0.01)}%;background:${COLORS[p]}"></span>`).join("");
  $("legend").innerHTML = parts.map((p) => `<span><i class="dot" style="background:${COLORS[p]}"></i>${p.replace("_", " ")}</span><span class="stat">${t[p].toFixed(2)} ms</span>`).join("");
}

$("feed").onclick = async (e) => {
  const iid = e.target.dataset.iid;
  if (!iid) return;
  const r = await fetch("/event", { method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify({ uid, iid: +iid, clicked: true }) });
  if (!r.ok) { toast("Event failed: is Kafka up?"); return; }
  toast("Click sent → Kafka");
  setTimeout(refresh, 400);  // give the worker a moment to update Redis
};

async function pollStats() {
  try {
    const s = await (await fetch("/stats")).json();
    $("ev").textContent = s.events_processed != null ? s.events_processed.toLocaleString() : "—";
    $("lag").textContent = s.lag_ema_ms != null ? `${s.lag_ema_ms.toFixed(0)} ms` : "—";
  } catch {}
}

$("rt").onchange = () => { prev = new Map(); refresh(); };
setInterval(() => { if ($("auto").checked) refresh(); }, 3000);
setInterval(pollStats, 2000);
loadUsers(); pollStats();
</script>
</body>
</html>


In [ ]:
%%writefile serving/worker.py
"""Stream processor: Kafka "events" -> Redis feature store.

Messages are keyed by user id, so all of a user's events land on one partition and are
applied in order. Each poll batch is written to Redis in a single pipelined round trip.
End-to-end lag (event produced -> features updated) is tracked for the demo and README.
"""
import json
import os
import time

import redis
from confluent_kafka import Consumer, KafkaError

from realtime_features import RedisStore

KAFKA = os.environ.get("KAFKA_BOOTSTRAP", "localhost:9092")
TOPIC = os.environ.get("EVENTS_TOPIC", "events")
REDIS_HOST = os.environ.get("REDIS_HOST", "localhost")


def main():
    r = redis.Redis(host=REDIS_HOST, port=6379)
    store = RedisStore(r)
    consumer = Consumer({
        "bootstrap.servers": KAFKA,
        "group.id": "feature-worker",
        "auto.offset.reset": "earliest",
    })
    consumer.subscribe([TOPIC])
    print(f"feature-worker consuming {TOPIC} from {KAFKA}", flush=True)
    lag_ema, total, last_log = None, 0, time.time()
    while True:
        # consume() waits for the full timeout unless the batch fills up, so keep it short for low lag
        msgs = consumer.consume(num_messages=500, timeout=0.05)
        batch, lags = [], []
        for m in msgs:
            if m.error():
                if m.error().code() not in (KafkaError._PARTITION_EOF, KafkaError.UNKNOWN_TOPIC_OR_PART):
                    print(f"Kafka error: {m.error()}", flush=True)
                continue
            ev = json.loads(m.value())
            batch.append((int(ev["uid"]), int(ev["iid"]), int(ev["clicked"])))
            if "sent_ms" in ev:
                lags.append(time.time() * 1000 - ev["sent_ms"])
        if not batch:
            continue
        store.record_many(batch)  # same record() semantics the offline backfill used
        total += len(batch)
        pipe = r.pipeline(transaction=False)
        pipe.incrby("rt:stats:events", len(batch))
        if lags:
            last = lags[-1]
            for x in lags:
                lag_ema = x if lag_ema is None else 0.95 * lag_ema + 0.05 * x
            pipe.set("rt:stats:last_lag_ms", round(last, 2))
            pipe.set("rt:stats:lag_ema_ms", round(lag_ema, 2))
        pipe.execute()
        if time.time() - last_log > 10:
            print(f"processed {total:,} events, lag_ema {lag_ema:.1f} ms" if lag_ema else f"processed {total:,}", flush=True)
            last_log = time.time()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile smoke_test.sh
#!/usr/bin/env bash
# Runs the whole pipeline on a small fake dataset (~1-2 minutes on a laptop).
# Run this first: if it passes, the real run will too.
set -euo pipefail
cd "$(dirname "$0")"

python tests/make_synthetic.py --out_dir data/synthetic/KuaiRand-Pure/data
python prepare_data.py --raw_dir data/synthetic --out_dir data/processed_synth
python baselines.py --data_dir data/processed_synth --results_dir results_synth
python train_retrieval.py --data_dir data/processed_synth --results_dir results_synth --epochs 3 --batch_size 512
python train_retrieval.py --data_dir data/processed_synth --results_dir results_synth --epochs 3 --batch_size 512 --no_side_features
python train_retrieval.py --data_dir data/processed_synth --results_dir results_synth --epochs 3 --batch_size 512 --no_logq
python train_ranking.py --data_dir data/processed_synth --results_dir results_synth --model dcn --epochs 2
python train_ranking.py --data_dir data/processed_synth --results_dir results_synth --model dnn --epochs 2
python build_rt_features.py --data_dir data/processed_synth
python train_rt_ranker.py --data_dir data/processed_synth --results_dir results_synth --epochs 2
python export_artifacts.py --data_dir data/processed_synth --out_dir artifacts_synth
python make_report.py --results_dir results_synth

echo
echo "Smoke test passed. Now download the real data and run ./run_all.sh"
echo "(Optional) test the serving stack on fake data: ARTIFACTS=./artifacts_synth ./demo.sh"


In [ ]:
%%writefile tests/make_synthetic.py
"""Generate a small fake dataset with the KuaiRand-Pure file layout and schema.

Used by smoke_test.sh to check the whole pipeline runs in about a minute, before the real
dataset is downloaded. Clicks depend on hidden user/item preferences, so models should beat
the baselines here too.
"""
import argparse
import os

import numpy as np
import pandas as pd


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_dir", default="data/synthetic/KuaiRand-Pure/data")
    ap.add_argument("--n_users", type=int, default=1500)
    ap.add_argument("--n_items", type=int, default=600)
    ap.add_argument("--rows", type=int, default=150_000)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()
    rng = np.random.default_rng(args.seed)
    os.makedirs(args.out_dir, exist_ok=True)

    n_topics = 8
    user_ids = np.arange(args.n_users) + 1000
    item_ids = np.arange(args.n_items) + 5000
    user_topic = rng.integers(0, n_topics, args.n_users)
    item_topic = rng.integers(0, n_topics, args.n_items)
    item_pop = rng.pareto(1.5, args.n_items) + 1

    users = pd.DataFrame({
        "user_id": user_ids,
        "user_active_degree": rng.choice(["high_active", "full_active", "middle_active", "UNKNOWN"], args.n_users),
        "is_lowactive_period": rng.integers(0, 2, args.n_users),
        "is_live_streamer": rng.integers(0, 2, args.n_users),
        "is_video_author": rng.integers(0, 2, args.n_users),
        "follow_user_num_range": rng.choice(["0", "(0,10]", "(10,50]", "(50,100]"], args.n_users),
        "fans_user_num_range": rng.choice(["0", "[1,10)", "[10,100)"], args.n_users),
        "friend_user_num_range": rng.choice(["0", "[1,5)", "[5,30)"], args.n_users),
        "register_days_range": rng.choice(["31-60", "61-90", "91-180", "181-365", "366-730"], args.n_users),
    })
    for i in range(18):
        # onehot_feat0 leaks the hidden topic, so side features genuinely help
        users[f"onehot_feat{i}"] = user_topic if i == 0 else rng.integers(0, 5, args.n_users)
    users.to_csv(os.path.join(args.out_dir, "user_features_pure.csv"), index=False)

    videos = pd.DataFrame({
        "video_id": item_ids,
        "author_id": rng.integers(0, 200, args.n_items),
        "video_type": rng.choice(["NORMAL", "AD"], args.n_items, p=[0.95, 0.05]),
        "upload_dt": "2022-04-01",
        "upload_type": rng.choice(["ShortImport", "Web", "Kmovie"], args.n_items),
        "visible_status": "public",
        "video_duration": rng.integers(5_000, 120_000, args.n_items).astype(float),
        "music_id": rng.integers(0, 300, args.n_items),
        "music_type": rng.choice([0, 4, 9, 11], args.n_items),
        "tag": [f"{t},{rng.integers(0, 60)}" for t in item_topic],  # first tag = topic
    })
    videos.loc[rng.random(args.n_items) < 0.02, "video_duration"] = np.nan
    videos.to_csv(os.path.join(args.out_dir, "video_features_basic_pure.csv"), index=False)

    dates = pd.date_range("2022-04-08", "2022-05-08").strftime("%Y%m%d").astype(int).to_numpy()

    def make_log(n, random_exposure):
        u = rng.integers(0, args.n_users, n)
        if random_exposure:
            it = rng.integers(0, args.n_items, n)
        else:  # production feed: popularity- and topic-biased exposure
            p = item_pop / item_pop.sum()
            it = rng.choice(args.n_items, n, p=p)
            match = rng.random(n) < 0.4
            same_topic = [np.flatnonzero(item_topic == t) for t in range(n_topics)]
            it[match] = [rng.choice(same_topic[t]) if len(same_topic[t]) else 0 for t in user_topic[u[match]]]
        logit = -2.0 + 2.2 * (user_topic[u] == item_topic[it]) + 0.3 * np.log(item_pop[it])
        click = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
        d = rng.choice(dates if not random_exposure else dates[dates >= 20220422], n)
        hm = rng.integers(0, 24, n) * 100 + rng.integers(0, 60, n)
        base_ms = pd.to_datetime(d.astype(str)).astype("int64") // 10**6
        return pd.DataFrame({
            "user_id": user_ids[u], "video_id": item_ids[it], "date": d, "hourmin": hm,
            "time_ms": base_ms + (hm // 100) * 3_600_000 + (hm % 100) * 60_000,
            "is_click": click, "is_like": 0, "is_follow": 0, "is_comment": 0, "is_forward": 0,
            "is_hate": 0, "long_view": click, "play_time_ms": click * 10_000, "duration_ms": 30_000,
            "profile_stay_time": 0, "comment_stay_time": 0, "is_profile_enter": 0,
            "is_rand": int(random_exposure), "tab": rng.integers(0, 4, n),
        })

    std = make_log(args.rows, False)
    std[std.date <= 20220421].to_csv(os.path.join(args.out_dir, "log_standard_4_08_to_4_21_pure.csv"), index=False)
    std[std.date > 20220421].to_csv(os.path.join(args.out_dir, "log_standard_4_22_to_5_08_pure.csv"), index=False)
    make_log(args.rows // 5, True).to_csv(os.path.join(args.out_dir, "log_random_4_22_to_5_08_pure.csv"), index=False)
    print(f"Synthetic KuaiRand-Pure written to {args.out_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tests/test_parity.py
"""Training/serving parity: the Redis-backed store must produce exactly the features the
offline backfill (in-memory store) produces for the same event sequence.

    ./demo.sh test          # runs inside the stack against the real Redis
"""
import os
import random
import sys

import numpy as np
import redis

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from realtime_features import FeatureLogic, InMemoryStore, RedisStore  # noqa: E402


def main():
    r = redis.Redis(host=os.environ.get("REDIS_HOST", "localhost"), port=6379, db=15)  # separate db
    r.flushdb()
    rng = random.Random(0)
    users, items = range(1, 31), range(1, 51)
    events = [(rng.randint(1, 30), rng.randint(1, 50), rng.random() < 0.3) for _ in range(5000)]

    mem, red = InMemoryStore(), RedisStore(r, prefix="test")
    for e in events[:2500]:
        mem.record(*e)
        red.record(*e)
    mem.record_many(events[2500:])
    red.record_many(events[2500:])
    assert all(mem.get_user(u) == red.get_user(u) for u in users), "user state differs"
    assert mem.get_items(items) == red.get_items(items), "item state differs"
    for a, b in zip(mem.get_item_stats(items), red.get_item_stats(items)):
        assert np.array_equal(a, b), "item aggregates differ"

    r.flushdb()
    red.load_from(mem)
    assert all(mem.get_user(u) == red.get_user(u) for u in users), "bootstrap load differs"
    for a, b in zip(mem.get_item_stats(items), red.get_item_stats(items)):
        assert np.array_equal(a, b), "bootstrap item aggregates differ"
    more = [(rng.randint(1, 30), rng.randint(1, 50), rng.random() < 0.3) for _ in range(3000)]
    mem.record_many(more)
    red.record_many(more)  # aggregates must stay exact when streaming continues after bootstrap
    for a, b in zip(mem.get_item_stats(items), red.get_item_stats(items)):
        assert np.array_equal(a, b), "item aggregates drift after bootstrap"

    emb = np.random.default_rng(0).normal(size=(51, 8)).astype(np.float32)
    emb /= np.linalg.norm(emb, axis=1, keepdims=True)
    logic = FeatureLogic(emb, np.arange(51) % 5, np.arange(51) % 7, 0.3)
    for u in users:
        assert np.allclose(logic.compute(mem, u, list(items)), logic.compute(red, u, list(items))), f"features differ for user {u}"
    r.flushdb()
    print("Parity test passed: online (Redis) features == offline (backfill) features")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_ranking.py
"""Train the click-through ranking model (DCN-v2) and evaluate AUC / GAUC / LogLoss.

Evaluated twice on the test period:
  test_standard  exposures chosen by Kuaishou's production recommender (biased toward what it liked)
  test_random    randomly exposed videos (unbiased) -- how the model ranks items the old system wouldn't show

    python train_ranking.py --model dcn
    python train_ranking.py --model dnn     # ablation: same network without cross layers
"""
import argparse
import copy
import os
import time

import numpy as np
import torch
import torch.nn.functional as F

from baselines import ranking_fields, ranking_matrix
from common import get_device, load_processed, ranking_metrics, save_results, set_seed
from models import DCNv2


@torch.no_grad()
def predict(model, X, device, batch=16384):
    model.eval()
    out = []
    for s in range(0, len(X), batch):
        xb = torch.as_tensor(X[s:s + batch], device=device)
        out.append(torch.sigmoid(model(xb)).cpu().numpy())
    return np.concatenate(out) if out else np.array([])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="data/processed")
    ap.add_argument("--results_dir", default="results")
    ap.add_argument("--model", choices=["dcn", "dnn"], default="dcn")
    ap.add_argument("--epochs", type=int, default=5)
    ap.add_argument("--batch_size", type=int, default=4096)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--emb_dim", type=int, default=16)
    ap.add_argument("--n_cross", type=int, default=3)
    ap.add_argument("--patience", type=int, default=1)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    set_seed(args.seed)
    device = get_device()
    meta, inter, rand, user_feats, item_feats = load_processed(args.data_dir)
    names, vocabs = ranking_fields(meta)
    print(f"[{args.model}] device={device}  fields={len(names)}")

    splits = {s: inter[inter.split == s] for s in ["train", "val", "test"]}
    X = {s: ranking_matrix(df, user_feats, item_feats) for s, df in splits.items()}
    y = {s: df.is_click.to_numpy().astype(np.float32) for s, df in splits.items()}
    print({s: f"{len(v):,} rows" for s, v in X.items()})

    model = DCNv2(vocabs, emb_dim=args.emb_dim, n_cross=args.n_cross, use_cross=args.model == "dcn").to(device)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr)

    Xtr = torch.as_tensor(X["train"])
    ytr = torch.as_tensor(y["train"])
    best, best_state, bad, history = -1.0, None, 0, []
    for epoch in range(1, args.epochs + 1):
        model.train()
        t0, total = time.time(), 0.0
        perm = torch.randperm(len(Xtr))
        for s in range(0, len(perm), args.batch_size):
            idx = perm[s:s + args.batch_size]
            xb, yb = Xtr[idx].to(device), ytr[idx].to(device)
            loss = F.binary_cross_entropy_with_logits(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        val = ranking_metrics(splits["val"].uid.to_numpy(), y["val"], predict(model, X["val"], device))
        history.append({"epoch": epoch, "train_loss": total / len(perm), **val})
        print(f"epoch {epoch}  train_loss {total / len(perm):.4f}  val_auc {val['auc']:.4f}  val_gauc {val['gauc']:.4f}  ({time.time() - t0:.0f}s)")
        if val["auc"] > best:
            best, best_state, bad = val["auc"], copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= args.patience:
                print("Early stopping (ranking models usually overfit after 1-2 epochs on ID embeddings).")
                break

    model.load_state_dict(best_state)
    out = {"model": args.model, "args": vars(args), "history": history}
    eval_sets = [("test_standard", splits["test"])]
    if rand is not None:
        eval_sets.append(("test_random", rand[rand.split == "test"]))
    for name, df in eval_sets:
        if len(df) and df.is_click.nunique() == 2:
            scores = predict(model, ranking_matrix(df, user_feats, item_feats), device)
            out[name] = ranking_metrics(df.uid.to_numpy(), df.is_click.to_numpy(), scores)
            print(f"TEST {name}:", {k: round(v, 4) if isinstance(v, float) else v for k, v in out[name].items()})

    os.makedirs("checkpoints", exist_ok=True)
    torch.save(model.state_dict(), f"checkpoints/ranking_{args.model}.pt")
    save_results(args.results_dir, f"ranking_{args.model}", out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_retrieval.py
"""Train the two-tower retrieval model and evaluate full-ranking Recall@K / NDCG@K.

    python train_retrieval.py                          # full model (side features + logQ)
    python train_retrieval.py --no_side_features       # ablation: ID embeddings only
    python train_retrieval.py --no_logq                # ablation: no popularity-bias correction
"""
import argparse
import copy
import os
import time

import numpy as np
import torch

from baselines import eval_retrieval, retrieval_truth, train_click_counts
from common import get_device, load_processed, save_results, set_seed, topk_metrics
from models import TwoTower


@torch.no_grad()
def recommend(model, uids, n_items, k, device, chunk=4096):
    """Exact maximum-inner-product search over the full catalog (7.5k items fits easily).
    At production scale this is where an ANN index (FAISS/ScaNN) goes."""
    model.eval()
    item_ids = torch.arange(1, n_items + 1, device=device)
    item_emb = model.item_tower(item_ids)
    out = []
    for s in range(0, len(uids), chunk):
        u = model.user_tower(torch.as_tensor(uids[s:s + chunk], device=device))
        top = (u @ item_emb.T).topk(k, dim=1).indices + 1  # back to 1-based item ids
        out.append(top.cpu().numpy())
    return np.concatenate(out)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="data/processed")
    ap.add_argument("--results_dir", default="results")
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--batch_size", type=int, default=2048)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--temperature", type=float, default=0.05)
    ap.add_argument("--out_dim", type=int, default=64)
    ap.add_argument("--patience", type=int, default=2)
    ap.add_argument("--no_logq", action="store_true")
    ap.add_argument("--no_side_features", action="store_true")
    ap.add_argument("--ks", type=int, nargs="+", default=[20, 50, 100])
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--tag", default=None, help="name suffix for the results file")
    args = ap.parse_args()

    set_seed(args.seed)
    device = get_device()
    meta, inter, _, user_feats, item_feats = load_processed(args.data_dir)
    n_users, n_items = meta["n_users"], meta["n_items"]
    tag = args.tag or ("two_tower" + ("_id_only" if args.no_side_features else "") + ("_no_logq" if args.no_logq else ""))
    print(f"[{tag}] device={device}")

    train_pos = inter[(inter.split == "train") & (inter.is_click == 1)]
    pairs_u = torch.as_tensor(train_pos.uid.to_numpy(), device=device)
    pairs_i = torch.as_tensor(train_pos.iid.to_numpy(), device=device)
    print(f"Training pairs (clicks): {len(train_pos):,}")

    counts = train_click_counts(inter, n_items)
    log_q = None
    if not args.no_logq:
        q = (counts + 1.0) / (counts + 1.0).sum()  # +1 so unseen items get a finite correction
        log_q = torch.as_tensor(np.log(q), dtype=torch.float32, device=device)

    model = TwoTower(n_users, n_items, user_feats, item_feats, meta["user_vocabs"], meta["item_vocabs"],
                     use_side_features=not args.no_side_features, out_dim=args.out_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr)

    val_uids, val_truths = retrieval_truth(inter, "val")
    max_k = max(args.ks)
    best, best_state, bad, history = -1.0, None, 0, []
    for epoch in range(1, args.epochs + 1):
        model.train()
        t0, total, n = time.time(), 0.0, 0
        perm = torch.randperm(len(pairs_u)).to(device)
        for s in range(0, len(perm), args.batch_size):
            idx = perm[s:s + args.batch_size]
            if len(idx) < 2:
                continue
            loss = model.in_batch_loss(pairs_u[idx], pairs_i[idx], args.temperature, log_q)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
            n += len(idx)
        val = topk_metrics(recommend(model, val_uids, n_items, max_k, device), val_truths, args.ks)
        score = val[f"recall@{50 if 50 in args.ks else max_k}"]
        history.append({"epoch": epoch, "loss": total / max(n, 1), **val})
        print(f"epoch {epoch:2d}  loss {total / max(n, 1):.4f}  val {({k: round(v, 4) for k, v in val.items()})}  ({time.time() - t0:.0f}s)")
        if score > best:
            best, best_state, bad = score, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= args.patience:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)
    test_uids, test_truths = retrieval_truth(inter, "test")
    test = eval_retrieval(recommend(model, test_uids, n_items, max_k, device), test_uids, test_truths, inter, args.ks)
    print("TEST:", {k: round(v, 4) for k, v in test["all"].items()})

    os.makedirs("checkpoints", exist_ok=True)
    torch.save(model.state_dict(), f"checkpoints/{tag}.pt")
    save_results(args.results_dir, f"retrieval_{tag}", {"model": tag, "args": vars(args), "test": test, "history": history})


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train_rt_ranker.py
"""Ranking with vs. without real-time features, plus an ablation over feature groups.

Trains the same network on identical data and seeds:
  base       categorical user/item/context fields only (what train_ranking.py does)
  all        + all 7 real-time features
  candidate  + only features that vary across candidates in one request (similarity, tag/author
             match, item recent CTR/volume). These are what can reorder a single user's feed.
  user       + only user-level features (user recent CTR / history size). Constant within a
             request, so they can raise global AUC without improving per-user ordering (GAUC).

The variant served online (exported as ranker_rt) is picked by validation GAUC, not test.

    python train_rt_ranker.py
"""
import argparse
import copy
import json
import os
import shutil
import time

import numpy as np
import torch
import torch.nn.functional as F

from baselines import ranking_fields, ranking_matrix
from common import get_device, load_processed, ranking_metrics, save_results, set_seed
from models import DCNv2
from realtime_features import FEATURE_NAMES

VARIANTS = {
    "all": FEATURE_NAMES,
    "candidate": ["sim_to_recent_clicks", "tag_match_recent", "author_match_recent", "item_recent_ctr", "item_recent_volume"],
    "user": ["user_recent_clicks", "user_recent_ctr"],
}


@torch.no_grad()
def predict(model, X, D, device, batch=16384):
    model.eval()
    out = []
    for s in range(0, len(X), batch):
        xb = torch.as_tensor(X[s:s + batch], device=device)
        db = torch.as_tensor(D[s:s + batch], device=device) if model.n_dense else None
        out.append(torch.sigmoid(model(xb, db)).cpu().numpy())
    return np.concatenate(out)


def fit(name, vocabs, data, args, device, cols):
    use_rt = len(cols) > 0
    set_seed(args.seed)
    cfg = dict(emb_dim=args.emb_dim, n_cross=args.n_cross, use_cross=args.model == "dcn", n_dense=len(cols))
    model = DCNv2(vocabs, **cfg).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=args.lr)
    Xtr, Dtr, ytr = data["train"]
    Xtr, Dtr, ytr = torch.as_tensor(Xtr), torch.as_tensor(np.ascontiguousarray(Dtr[:, cols])), torch.as_tensor(ytr)
    best, best_state, bad = -1.0, None, 0
    for epoch in range(1, args.epochs + 1):
        model.train()
        t0, total = time.time(), 0.0
        perm = torch.randperm(len(Xtr))
        for s in range(0, len(perm), args.batch_size):
            idx = perm[s:s + args.batch_size]
            if len(idx) < 2:  # BatchNorm needs >1 row
                continue
            logits = model(Xtr[idx].to(device), Dtr[idx].to(device) if use_rt else None)
            loss = F.binary_cross_entropy_with_logits(logits, ytr[idx].to(device))
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(idx)
        Xv, Dv, yv, uv = data["val"]
        val = ranking_metrics(uv, yv, predict(model, Xv, Dv[:, cols], device))
        print(f"[{name}] epoch {epoch}  train_loss {total / len(perm):.4f}  val_auc {val['auc']:.4f}  ({time.time() - t0:.0f}s)")
        if val["auc"] > best:
            best, best_state, best_val, bad = val["auc"], copy.deepcopy(model.state_dict()), val, 0
        else:
            bad += 1
            if bad >= args.patience:
                break
    model.load_state_dict(best_state)
    return model, cfg, best_val


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", default="data/processed")
    ap.add_argument("--results_dir", default="results")
    ap.add_argument("--model", choices=["dcn", "dnn"], default="dnn")
    ap.add_argument("--epochs", type=int, default=5)
    ap.add_argument("--batch_size", type=int, default=4096)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--emb_dim", type=int, default=16)
    ap.add_argument("--n_cross", type=int, default=3)
    ap.add_argument("--patience", type=int, default=1)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--variants", nargs="+", choices=list(VARIANTS), default=list(VARIANTS))
    args = ap.parse_args()

    device = get_device()
    meta, inter, rand, user_feats, item_feats = load_processed(args.data_dir)
    rt_std = np.load(os.path.join(args.data_dir, "rt_standard.npy"))
    rt_rand = np.load(os.path.join(args.data_dir, "rt_random.npy")) if rand is not None else None
    names, vocabs = ranking_fields(meta)

    def pack(df, rt, with_uid=False):
        X = ranking_matrix(df, user_feats, item_feats)
        y = df.is_click.to_numpy().astype(np.float32)
        return (X, rt, y, df.uid.to_numpy()) if with_uid else (X, rt, y)

    data = {}
    for s in ["train", "val", "test"]:
        mask = (inter.split == s).to_numpy()
        data[s] = pack(inter[mask], rt_std[mask], with_uid=s != "train")
    if rand is not None:
        mask = (rand.split == "test").to_numpy()
        data["test_random"] = pack(rand[mask], rt_rand[mask], with_uid=True)

    out = {"model": args.model, "args": vars(args), "features": FEATURE_NAMES, "variants": {}}
    os.makedirs("checkpoints", exist_ok=True)
    runs = [("base", [])] + [(v, [FEATURE_NAMES.index(f) for f in VARIANTS[v]]) for v in args.variants]
    for name, cols in runs:
        model, cfg, val = fit(name, vocabs, data, args, device, cols)
        res = {"features": [FEATURE_NAMES[c] for c in cols], "val": val}
        for split in ["test", "test_random"]:
            if split in data:
                X, D, y, u = data[split]
                res[split] = ranking_metrics(u, y, predict(model, X, D[:, cols], device))
                print(f"[{name}] {split}: AUC {res[split]['auc']:.4f}  GAUC {res[split]['gauc']:.4f}")
        out["variants"][name] = res
        tag = "base" if name == "base" else f"rt_{name}"
        torch.save(model.state_dict(), f"checkpoints/ranker_{tag}.pt")
        with open(f"checkpoints/ranker_{tag}.json", "w") as f:
            json.dump({**cfg, "rt_feature_idx": cols, "field_names": names, "vocabs": vocabs}, f)

    # serve the real-time variant with the best *validation* GAUC (per-user ordering is what a feed needs)
    rt_names = [n for n, _ in runs if n != "base"]
    chosen = max(rt_names, key=lambda n: out["variants"][n]["val"]["gauc"])
    out["served_variant"] = chosen
    for ext in ["pt", "json"]:
        shutil.copy(f"checkpoints/ranker_rt_{chosen}.{ext}", f"checkpoints/ranker_rt.{ext}")
    print(f"\nServing variant (best val GAUC): {chosen}")
    b = out["variants"]["base"]
    for n in rt_names:
        v = out["variants"][n]
        print(f"  {n:10s} test AUC {b['test']['auc']:.4f} -> {v['test']['auc']:.4f}   GAUC {b['test']['gauc']:.4f} -> {v['test']['gauc']:.4f}")
    save_results(args.results_dir, "ranking_realtime", out)


if __name__ == "__main__":
    main()


## 2. 檢查環境

In [ ]:
!chmod +x *.sh
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())

## 3. Smoke test（假資料，約 2 分鐘）
有錯誤的話，把訊息貼給 Claude。

In [ ]:
!./smoke_test.sh

## 4. 下載 KuaiRand-Pure（約 200MB）

In [ ]:
!mkdir -p data && cd data && curl -sL -O https://zenodo.org/records/10439422/files/KuaiRand-Pure.tar.gz && tar -xzf KuaiRand-Pure.tar.gz && ls -R | head -20

## 5. 正式實驗（retrieval、ranking、real-time features、匯出 artifacts）

In [ ]:
!./run_all.sh

## 6. 結果

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('results/RESULTS.md').read()))

## 7. 下載（repo + README + results + artifacts）

In [ ]:
!cd /content && zip -rq kuairand-recsys-results.zip kuairand-recsys -x 'kuairand-recsys/data/*' 'kuairand-recsys/results_synth/*' 'kuairand-recsys/artifacts_synth/*' 'kuairand-recsys/checkpoints/*' '*/__pycache__/*' && ls -lh kuairand-recsys-results.zip
from google.colab import files
files.download('/content/kuairand-recsys-results.zip')